<div style="border: 2px solid #8A9AD0; margin: 1em 0.2em; padding: 0.5em;">

# Filter, plot and explore single-cell RNA-seq data with Scanpy (Python)

by [Morgan Howells](https://training.galaxyproject.org/hall-of-fame/hexhowells/), [Wendi Bacon](https://training.galaxyproject.org/hall-of-fame/nomadscientist/)

CC-BY licensed content from the [Galaxy Training Network](https://training.galaxyproject.org/)

**Objectives**

- Is my single cell dataset a quality dataset?
- How do I generate and annotate cell clusters?
- How do I pick thresholds and parameters in my analysis? What's a "reasonable" number, and will the world collapse if I pick the wrong one?

**Objectives**

- Interpret quality control plots to direct parameter decisions
- Repeat analysis from matrix to clustering
- Identify decision-making points
- Appraise data outputs and decisions
- Explain why single cell analysis is an iterative (i.e. the first plots you generate are not final, but rather you go back and re-analyse your data repeatedly) process

**Time Estimation: 3H**
</div>


<h1 id="install-libraries">Install libraries</h1>
<p>This tutorial requies some libraries to be installed which is done below (igraph and louvain are not used directly and are just required for plotting). The <code class="language-plaintext highlighter-rouge">-q</code> parameter hides most of the outputs of the installation in order to make the notebook a bit cleaner. If there are any issues with the installation, then removing this parameter may give you more information about the issue.</p>


In [ ]:
pip install scanpy -q

In [ ]:
pip install igraph -q

In [ ]:
pip install louvain -q

In [ ]:
pip install pandas -q

<hr />
<p>We can now import the two libraries that we will be using, <strong>scanpy</strong> is the primary library that we will use and will handle all the plotting and data processing. Meanwhile, <strong>pandas</strong> is used briefly for some manual data manipulation.</p>


In [ ]:
import scanpy as sc
import pandas as pd

<h1 id="load-data">Load Data</h1>
<p>You can import files from your Galaxy history directly using the following code. This will depend on what number in your history the final annotated object is. If your object is dataset #1 in your history, then you import it with the following:</p>


In [ ]:
mito_counted_anndata = get(1)                   # get an object from Galaxy history
adata = sc.read_h5ad(mito_counted_anndata)      # read in the file as h5ad object

<p>Alternatively, if you don’t want to get the dataset from your Galaxy history, you can also download the input file from Zenodo, running the code below:</p>


In [ ]:
%%bash
wget -nv https://zenodo.org/record/7053673/files/Mito-counted_AnnData

In [ ]:
adata = sc.read_h5ad("Mito-counted_AnnData")

<h1 id="filtering">Filtering</h1>
<p>You have generated an annotated AnnData object from your raw scRNA-seq fastq files. However, you have only completed a ‘rough’ filter of your dataset - there will still be a number of ‘cells’ that are actually just background from empty droplets or simply low-quality. There will also be genes that could be sequencing artifacts or that appear with such low frequency that statistical tools will fail to analyse them. This background garbage of both cells and genes not only makes it harder to distinguish real biological information from the noise, but also makes it computationally heavy to analyse. These spurious reads take a lot of computational power to analyse! First on our agenda is to filter this matrix to give us cleaner data to extract meaningful insight from, and to allow faster analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>What information is stored in your AnnData object? The last tool to generate this object counted the mitochondrial associated genes in your matrix. Where is that data stored?</li>
<li>While you are figuring that out, how many genes and cells are in your object?</li>
</ol>
<blockquote class="tip" style="border: 2px solid #FFE19E; margin: 1em 0.2em">
<div class="box-title tip-title" id="tip-hint"><button class="gtn-boxify-button tip" type="button" aria-controls="tip-hint" aria-expanded="true"><i class="far fa-lightbulb" aria-hidden="true" ></i> <span>Tip: Hint</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Inspect the Anndata object by printing it with:</p>
<div class="language-plaintext highlighter-rouge"><div><pre style="color: inherit; background: transparent"><code style="color: inherit">print(adata)

print(adata.obs)

print(adata.var)
</code></pre></div>    </div>
</blockquote>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution"><button class="gtn-boxify-button solution" type="button" aria-controls="solution" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>If you examine your AnnData object, you’ll find a number of different quality control metrics for both cells (<strong>obs</strong>) and genes (<strong>var</strong>).
<ul>
<li>For instance, you can see a <code style="color: inherit">n_cells</code> under <strong>var</strong>, which counts the number of cells that gene appears in.</li>
<li>In the <strong>obs</strong>, you have both discrete and log-based metrics for <code style="color: inherit">n_genes</code>, how many genes are counted in a cell, and <code style="color: inherit">n_counts</code>, how many UMIs are counted per cell. So, for instance, you might count multiple GAPDHs in a cell. Your <code style="color: inherit">n_counts</code> should thus be higher than <code style="color: inherit">n_genes</code>.</li>
<li>But what about the mitochondria?? Within the cells information <strong>obs</strong>, the <code style="color: inherit">total_counts_mito</code>,  <code style="color: inherit">log1p_total_counts_mito</code>, and <code style="color: inherit">pct_counts_mito</code> has been calculated for each cell.</li>
</ul>
</li>
<li>You can see by printing the object that the matrix is <code style="color: inherit">31178 x 35734</code>. This is <code style="color: inherit">obs x vars</code>, or rather, <code style="color: inherit">cells x genes</code>, so there are <code style="color: inherit">31178 cells</code> and <code style="color: inherit">35734 genes</code> in the matrix.</li>
</ol>
</details>
</blockquote>
<h2 id="generate-qc-plots">Generate QC Plots</h2>
<p>We want to filter our cells, but first we need to know what our data looks like. There are a number of subjective choices to make within scRNA-seq analysis, for instance we now need to make our best informed decisions about where to set our thresholds (more on that soon!). We’re going to plot our data a few different ways. Different bioinformaticians might prefer to see the data in different ways, and here we are only generating some of the myriad of plots you can use. Ultimately you need to go with what makes the most sense to you.</p>
<h2 id="creating-the-plots">Creating the Plots</h2>


In [ ]:
# Violin - genotype - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-genotype-log.png'
)

In [ ]:
# Violin - sex - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='sex',
  save='-sex-log.png'
)

In [ ]:
# Violin - batch - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='batch',
  save='-batch-log.png'
)

In [ ]:
# Scatter - mito x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='pct_counts_mito',
  save='-mitoxUMIs.png'
)

In [ ]:
# Scatter - mito x genes
sc.pl.scatter(
  adata,
  x='log1p_n_genes_by_counts',
  y='pct_counts_mito',
  save='-mitoxgenes.png'
)

In [ ]:
# Scatter - genes x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='log1p_n_genes_by_counts',
  color='pct_counts_mito',
  save='-genesxUMIs.png'
)

<h2 id="analysing-the-plots">Analysing the plots</h2>
<p>That’s a lot of information! Let’s attack this in sections and see what questions these plots can help us answer.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-variation"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch Variation</div>
<p>Are there differences in sequencing depth across the samples?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret it?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-1"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-1" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>The plot <code style="color: inherit">violin - batch - log</code> will have what you’re looking for!
<figure id="figure-1" style="max-width: 90%;"><img src="...
RU5ErkJggg==
"" alt="Violin - batch - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-batch-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 1</strong>:</span> Violin - batch - log (Raw)</figcaption></figure>
</li>
<li>Keeping in mind that this is a log scale - which means that small differences can mean large differences - the violin plots probably look pretty similar.
<ul>
<li><code style="color: inherit">N703</code> and <code style="color: inherit">N707</code> might be a bit lower on genes and counts (or UMIs), but the differences aren’t catastrophic.</li>
<li>The <code style="color: inherit">pct_counts_mito</code> looks pretty similar across the batches, so this also looks good.</li>
<li>Nothing here would cause us to eliminate a sample from our analysis, but if you see a sample looking completely different from the rest, you would need to question why that is and consider eliminating it from your experiment!</li>
</ul>
</li>
</ol>
</details>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-biological-variables"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Biological Variables</div>
<p>Are there differences in sequencing depth across sex? Genotype?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret the <code style="color: inherit">sex</code> differences?</li>
<li>How do you interpret the <code style="color: inherit">genotype</code> differences?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-2"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-2" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Similar to above, the plots <code style="color: inherit">violin - sex - log</code> and <code style="color: inherit">violin - genotype - log</code> will have what you’re looking for!
<figure id="figure-2" style="max-width: 90%;"><img src="...
EARBEARx5G9KaN7GkrDKBAAAAABJRU5ErkJggg==
"" alt="Violin - sex - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-sex-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 2</strong>:</span> Violin - sex - log (Raw)</figcaption></figure>
<figure id="figure-3" style="max-width: 90%;"><img src="...
Lg0powAAAABJRU5ErkJggg==
"" alt="Violin - genotype - log. " width="1453" height="297" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-genotype-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 3</strong>:</span> Violin - genotype - log (Raw)</figcaption></figure>
</li>
<li>There isn’t a major difference in sequencing depth across sex, I would say - though you are welcome to disagree!
<ul>
<li>It is clear there are far fewer female cells, which makes sense given that only one sample was female. <em>Note - that was an unfortunate discovery made long after generating libraries. It’s quite hard to identify the sex of a neonate in the lab! In practice, try hard to not let such a confounding factor into your data! You could consider re-running all the following analysis without that female sample, if you wish.</em></li>
</ul>
</li>
<li>In <code style="color: inherit">Violin - genotype - log</code>, however, we can see there is a difference. The <code style="color: inherit">knockout</code> samples clearly have fewer genes and counts. From an experimental point of view, we can consider, does this make sense?
<ul>
<li>Would we biologically expect that those cells would be smaller or having fewer transcripts? Possibly, in this case, given that these cells were generated by growth restricted neonatal mice, and in which case we don’t need to worry about our good data, but rather keep this in mind when generating clusters, as we don’t want depth to define clusters, we want biology to!</li>
<li>On the other hand, it may be that those cells didn’t survive dissociation as well as the healthy ones (in which case we’d expect higher mitochondrial-associated genes, which we don’t see, so we can rule that out!).</li>
<li>Maybe we unluckily poorly prepared libraries for specifically those knockout samples. There are only three, so maybe those samples are under-sequenced.</li>
<li>So what do we do about all of this?
<ul>
<li>Ideally, we consider re-sequencing all the samples but with a higher concentration of the knockout samples in the library. Any bioinformatician will tell you that the best way to get clean data is in the lab, not the computer! Sadly, absolute best practice isn’t necessarily always a realistic option in the lab - for instance, that mouse line was long gone! - so sometimes, we have to make the best of it. There are options to try and address such discrepancy in sequencing depth. Thus, we’re going to take these samples forward and see if we can find biological insight despite the technical differences.</li>
</ul>
</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<p>Now that we’ve assessed the differences in our samples, we will look at the libraries overall to identify appropriate thresholds for our analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-genes"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: genes</div>
<p>What threshold should you set for <code style="color: inherit">log1p_n_genes_by_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-3"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-3" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">log1p_n_genes_by_counts</code> would do here, actually! Some people prefer scatterplots to violins.
<figure id="figure-4" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAY1BMVEX///+B
gYGEhITt7e0zMzP4+Pjz8/P+/v6AgID8/PyIiIjn5+fh4eF7e3uVlZXGxsbO
zs7W1taNjY2+vr63t7eqqqqjo6OcnJzb29uxsbErKys/Pz9vb29OTk5eXl4d
HR0JCQkUMmaoAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAgAElEQVR42u1diWKq
OhBFQhJDwhpW1/7/V74zCSBaq7a173aB99rbKlLNMDNnziwJgnccMuDBcnzx
wQP13peoZdX+j+PdyywXdfkf1GX89r6XLMdXH++9+6VY1uz7uZhFWf4voSjg
q/dgMb2s2f9wiPdqgNhsouX48uMoriMy5R+Vw9fsiN4L9uQCrd9/RG/aq0BK
9/3Mr8hFLv9WLm5NFUX2MvikvgwCXo6nyIVURQ0rC2xwuu/fJxfp5bIcz5KL
8jc5n1DbaI2i9/+RRS5PlMvkVrgXjlIfsWOD0i3HE+UiA8W2x2gdrI+bqB9F
9U65KL4I5slyUQE3WdBtbRnP+M13yoVzvliy58lFeRcD0QRbm5QzjnP7PtDL
cjF4f1xuwcufx8ne77Ogikx43OwM/crbzeHl3YzCTBqLXD6rL94tyCDfN4Fm
QbIbH99+BI6NTmaxZ0+JK3G3H1buZ3Mcb/fNO8MRJ5FOcGcXl+MpPEzAD6Qn
jAfxbgxkPqQvepHL8/QFAilejlHUHKLNvnC3vno/TvaCWezY0/y+GiDYrPyF
lnXzXv+tFr//XH2RAyM2isVzmNEH6zoWnPw8O/acVyzHIpdFLsuxyGWRyyKX
RS7LschlkctyLHJZ5LLIZZHLcixyWeSyHItcFrksxyKXRS6LXBa5BEtBxqIv
i1yW8Rk/X184XwzZd5MLim6U1otcvp2+QCSMLYL4fnJRi4t5dx+fa046f2yz
jCP55/3I8krr6oKT/3E9/yQLdaZAi1z+tVxOwuDBabhVtGCnf+v30ZbE9Xaz
3QV6fzzkY6P4dmmX+LdyobYk1gm9z3apSsLFv3wXvz/MuYiyTRGYiHur9oH+
yuV4cv+L+9ZFrrfyxT3UbvYvy6L987gSCqK3tZPJyxi2PMmOKSPOo0t5hs+l
Z//VXyU27+kLOyQIJbvAbNwD6lly4dyw4HyEo1R+KNYkITQQim7Rl2v9+4L6
kUWSBvH6g3NI3jyEukH4Kz9XSwk9WNNFLmdrV7xsomPWbY974GS/lNFTKctz
mUjBT1O2/nQW8878sWn6JWb5imfqCx8HnZzofghCaIwKGCdjneY3yEUur0b4
8nG6svzQfMvHcQCZLsXEvNHfexz1FwUTvS2T2TwENbtpn2nHlDrzKGfPDH9P
L3bsdfe+uuILoq/J6is351RezpFTvQiMUYtcnvWKu/Ng3tpFRp3N9YXb4ULw
Pzcq86vkwmfjSt9TCqMu7JsLMy8uo9Qil4+/gp+lOl89K1XwyM4AMvibo2W/
Mr/P7lAo14CWvBggO/h/eS4atcjlg6+ANrBK3JHam1ypmgMz9UrVFn358Cuk
n2z59vOseMPICSFfT3W6GLv5+2f/fh0e47fLZri+7iy4MfzG5Hhn1nDOIpcP
vkLddgI0xPyNUeRnbucKa4nnmfhrcpHO+vDvU8+vgseA22/XF/7t+izkX0yM
Ra9uT7bbRjv2XeQi/2h15+tV3od5Ee+/j1xUcGVzoD+oL9FUhPxd+sWW/D6F
D9sqCLLo28hF3oV2f8SO2e3xGOXfSS6YQr/IJTCKoVjlG9kxqEvG/7xcZHSn
SCy6H2/I1xPGB/r4cf8tzS37xf+aXIr6WNd9+jm5XOZK2MCgvGM1FfgzeY2p
cWUAf49P7g8vu/0u6T9hx9SFbKbflHnPbc5u7H7yF3l+eweZRvf9tLo0OMKX
nHfiXdhYXUlEsz/qX0qeHHa73eGjciEuUmp+tpJKWeYkJB6/0d8ixSrtmeo/
Z8eCvm3auvmEHZOimN3V3KrhLv/IPsnqDX2Rf8+O3Suhj97pGtiH5vO87dr5
H81XZodNtPkEDzPsrjjiO8tOZRjP89d/UF+OtjC6+LjfVyN5QvJRuR71T91e
WulPoxBHebk+uL+ceoinkT9Mlq/zLwdztj/yKye7eaC8VU7uRLxlltT5yAaf
u5Tc/8T6S7moN/yNCh6jm1XwszY2j14F0m3S9HXtF0nsjhtF2yNvKv5wh9n0
k5hXnF8NKuWrkuihSl34JXxgISV/fEDQz86/bPeHw4iTK4sTduW7+TEqNA4K
Ndbpu6DSsMt8NT8TivIlSGpowrgQnHStGNfDpQfKY3ws9HM2N3+1ymzPZgYi
YDBbcTnbkE/e3x9Zud4ZL5fBPhEC6O381la2u4bilLyOB4eH5TVz+Gih30+i
b67oSzdvOinQi7w7bvfGj/Bp7u6PLMfPf7JjQ1HLWTzo1SM3p5exjClPeA63
/6XheSOeJHekHuyv+Tl85xU8hvTLJvL2RAV6Q9u9snI3fnR1W1+G7gwx+n05
ra+8F+joWTm/IyfP7m+F6XFXbnjlxPtok4y2Pzj/0hlDdXNuH3DVefhVbKaF
jR6Y7iryESePLuByRbV3KOqMSOOz6BNV/PzcbL0xPU7J4AHaTfnCZ/aT65S8
Ddu4RSu2OigCnh7ECJWiR7YPZzMrpK7BNVA1Ut3Mp/Cu4287DmX56yvzX11v
yf3dTYqid8eXbThsj8wf6a8c4jdyEU4yA2vibNIFGtKGvx2+S+9PkLPJrv8h
cy5nFQTid+crlXfJ/hlxCtX5Q3Zsho3UuIXvGzCWMS+yztwkzrrrOuOD0plc
RKd+eb2ll8TmDQsUPQbH5FB2r5ToDJcyGHuLX7ltzE64svTUYKG8i7pmzuQU
go7H75bLZFr4/gRyhv5G+WC87/tXaZf4jo8vf7tuUtziDdTNhg1+nl74TXzm
1Xxl3WqyH9fbVKPHRu1LP71Cjo3m8jy+H7WKBzm7tp6jXKheQ1l+ncG53EFe
/WY7tufZdrd/my+MHqSh5NlUWc6LKyN4eJErzd+S8IQTzDV9kvIVrfyr8Vik
2vU5aazel3/BgsrgjN13K9jl6pV/Fxl7o1VG+hePncnquqXjvzYrc4WHSaPu
CfXJlHvhs8THeZw4uG0VyCnkVHLOPMu7Zkn96iqy16tclWlQlE+Qi/dQ0mMl
h68mEE21GDo4je45fZcTPyPvKcCvbrJ8vcoN4ZrySX3iJ7mc/AH8AOpa2am2
ValxSNAgDJ7752606/Ffnky+Us8/BZWflcvMb/DLYJ3WtCtmAb/Wr54XXcdv
56vVn5FLc3jZbw/b7TP6xM8nlcszvzKSCdLH/TO5zG3b242V3uvr7q/IxTSH
PrOZeoZczhojCaTR8Mqg06MqDXSzdkj5ZMfG+B4P8Bu9+vQipv+Of9F3GMDo
I9VEYxoGX5pN1/d4WpzlhL0/ktPEM95d5+aZZX+rH7navBxfjk/qf1Gn8okZ
8Z/NKlik6fgFTgYu0JO5Mra7bsfYr3b8V/KVxScnLp/pS+doeqvH7gjuW59O
lD/nYlIWfqFqTM8rz6af5pF+UXgZS/nLeX6+F5+sg5W+7mLaCMkHgGqM/AeS
WF2youq1gIEF+AzayanMYgb0xBQTZeIXKdDrVT7s0rqvPqwv6gIZgwIbfxuq
9vwan3LyhZlzx5ObG+fHcHmaTuLrbC6nXvlEjDS/247t9ofd4fAp/yKBoWj6
jheHCqZ7OhATFD7NRhRCXHEVTKtX5Zmj+NS88GyWBlW/vp4/+FS/2NCDNPaw
+LZVelhYMVQ/eP/BhyoVHqi3/zgn4NWTYLjPcau5K1ETzSnlb+aTGdr4DjT9
/cP9lQ7njpVK3k+cNvqhpRO5UFQPE3Sd9JBLvX1PcNWhPpDYNEUWUV0vmg74
Lwv+X69yX1flJ+TiModdd8b4Dt7F8y7c3e5QpxPzwoMRTZ0NBh6Q2vBydzU4
fXZ1kA8FoepX2zGyKWz7uX5kMeZPZktNo5HP3EjWz4AYt64vPDvD2PmQ7nIp
TRIP/V+JV2OdXcKHB78pMxZdXdr8afNhXHe4cqYHelKwaXY5v7BcOqNEc3OZ
BiNUzAWXM+cu/8LImNd8MoL9l039tHkXVDnGBjUqitNaB1d3UvBV99oMwaKa
4N0ZV/Mn5yfzO3nA6N2Nq8gWu6DDTa4cFYJbfSrTgG+fx/vsAjhL+edGkF1Z
5X636p84r0+OUQu3Yk6iIJaXU7ip3qxWGhqUxjOGDRX+YH/lbtu2+93z7Jgz
UHxeKeahNGyayudzY2fbyzJv8JgZKy+UNDn/SF/zL6qHgTcQ0RPlMuHa3vq0
C7UJBAJGTfdOG0TRnZfESGZcvM/MLO3PzzbY+nt+P6IEzOapfn90GxOOZdnJ
kdHT3eu+FK2n8Ic4UKY8PFbaiD+Jx4LmeDgc62f6fU/HKONiI36iBK5UtKrL
AF759mbVZZX/TRC4e6MV5nf7fVP1UyR+OG5oC8vNVk9V4dHV21XeItfUpCO+
npY8iB1iz3kyZWiiVCNUULUZ9UrN9U8RsjtBNHV7bukZTrjrmeR3lUtNA3ZG
fcmoHzlpdZuc5sYFb5VzXaHfiXQszp0Nz/IzQkvNWsm7qbDAZywLfX0yirpo
W+YfHXv+xr4N8hvKZaPmwxPY0WUwqf11KAnb3qy0mK1IR3QxssQ9m0wTCDLe
nxXLqGHjvzHoV7MeWX7inOXlGC0gPJYPBWrYxu/2zHkRXN+77Bt3LUdXH1GT
XKix8ijH7ZF5/XY/8kWfauAKwqXLGgdjf2Q/FvWzU/nrqQBgCPod89INU/h5
p9+6CbpiahW4PXAO9ME7BgYR4FDfcQ7JKivWU17M4AToijpOea3ollnmVwvI
/KQQUMjdmNPv58Wy/LzOjNw85VyUY6XVaK0GXz+Dd/yRHU0cVLjs5LyBtb+r
XAKdRNvd1KxiBju2mbZLjN52tdxMfLBgM6lgLlw19dBguknNpzSAyPnJq8tJ
b+Q8JzZsPatfb3fxoBrIebe1UD9h+kJ0teGL1sF5elmgbWyd6vZEAGzfkMr5
3ILubG9DJFvwIuLtKf7vurybDZG5voTaPWGQQvOw2pgzzoCpx4bDyosBQZzd
Jgy+r98fpRPRt8PxJUrNNor0+b6i8vrneTUihE9oCzBZXKygkrxXp/tfDijA
dyTl4rTmOCX3tcxU9OIeqWbNGLfGXcjL/TB4cHvbEvnd9+XZvGGsokcmhg46
ZTIkgXXmul8EsS+jMo0nuy3I2OXC6HzQJXtpM6W4SFLS95xdAdNnOPoTlgu6
VnT8e/Tv35wzFj0ilbOiCeb5FpVXKsjoM467IA/dFPUrayhONbkX0WPN5w2z
l+2Vl/bpkbpydbsxQLkCN/Vt9AV9ltfvtttyofEiaqy0mBhiP7WvQ92LsFPf
S9dkgTNLObyHLwE/xZhiqJGhr9yB5tyxmDPdyvUArUVwbagv5eOy7CFYMMMr
NxsT/i0/NnxrP153gQVRZzsdFbkbESuoJmkK4UEjY0yMW1SeB9NWbqKbyted
7mRD0YEZzpBehJ1gTnKdHX3OGQBQN3rQr7gTcWcw5Dfgkw/MDU16c8TNHTvG
g7HwYpSKg0AdO9Us+RAic56dw7aBIVbgXbg2PukyrTKKm7n08y5qOpkmy+FL
02CzweZz2zhIzl9DswfnnEp5C36pfySaV/OT1y/J+rD7FM8/WWz/cfTY/0rr
LjLjP25Vi/FOLZxuiOpsoB+92Dpv0tFwJkOlZlQ7qxmfEV+y57P5Kez542Xl
PwJpl6ucp1HZtv3H9xdT0yfRrwfCeD6S+/hk2FkUef+CbNLJlrChSFZ6eOBM
XUaW0dc0s6t7/cqx6/x0nz84T5bfTn8q9W36kgJxowAyerB/j9fFfGUGdtKP
6IflyDS3HTPucc1mfleZXGW5OLdLeXEaPTNWnJ+rl/Pzl+9XPmuPrG/g97fM
N79+Ni+mpw6V2XhY1jIX66dIwGibFfmVqJ0SkoWP9uX5TSvSecG41uqVF9Dm
lUQu3IxSP6LC/Dqf/OG5CiceprBjMb/w1o2iFzGgAISaVRd0MJdjN54kqXl3
UjFhBkoM2Ew75947T1RwD/b42EjjaTgYsAJeiOIZZs5jqAG23+T9f4hcaN+3
6gP7JBI9f4rjFJuNC3WsPBKUCoKAiYKL5+MWFyMMOLkXSIh1vmCMqwwgGLCB
5pNie/LCjZgf65rdXMBugHojRcxfDyqVF/Env89dfs9936Iosh+SixslNvZQ
doMNMUPNUZ+ZDpG9C0Nm6zROeRUUN1VX5lyJdOwa82MWdOEjTjmmyCbbxUma
viSKyzGAEa9zlj9TX3iyv1U+9rZc1LXRq2riS8CTVTlnTad5O21Ewge7U7GL
wUlTGzmyAmKY20f5GKteTQ8f6pmuzGJU6KzpzovJXTma/JFze1Z9lXzE70t1
xtx6VswBK9O5Va7gP/JeW+BfSlxSwWVWean14tJFqTFKJRnTQvqo3sxjc+c6
ikIi5p/NHafk5NiNxnjHrw37kz/Qv1Dw9m65qEsYxIeMPvEuSJ5QvpJSKghU
8kLkGaxW3WfdWNfCgtOUBJ6Zcxgl8jkizs/31mS5OWUDKPNGbUwYmM2HUOqi
1FkF6kfsg32l7xXetfjI/shKXQlkjKMp/aZJIxqri4oPAGu2b6jO8srjX/xW
DzVnboB8Tq0APsdLjiv3IHmI+wt9EdP6QmZ+K/b4iXLh0csRrRabrfpsXZ+c
yDGT9fkgumxYOAW0OxivulJVRdqSFdkYL3I165IKQLbQM1CwcfER2hOuqInx
nLcKzhE7vyIPcWq8lT9MXwrMG++Mzj8ZV45oyZKZ4arhBRuqvmytCB8zl4uh
IFYPliXH6nr0LPOzm3pIdBauFoBnaqxHrzypg8Xmp4k2amgu4xV7VXQ2pvfV
OBpF/ax+saf1v6hT3yRZfJMjDDHGA9qgyLKugpyqsVXfFHrAW6Aq2ZXog4Yp
u41g6eGeIN8wL2s23Jzlp550+WorgGsX/UlyeXOs5MNyUWZK72p3r7vAM4eS
NKDGfKwOWtmTAJwzh4sbtOyhP5lWtW8zMdKgU7GyHSgeasRgp7z+mNcaJwJq
p2FyPgDN6invLC/Lan6IXG5rzsNyoTjcjbBygy6H0JCaXXt06uHZmoourOUU
NuqiyjO3UrZXQ3TIKd3swJrQppuHK5lR5vLdiTEJX1DiRr/igYl0y1xNhxwd
zHcWTfRFrxg2PprvZOnSwA14ZMOyNstYTpRYX7MevqyweWFquP7zXV6kIRPV
YSO6bgwzqSWDFT7prOf2EhkESRxpofNpaC9/CwsMm/mqn6kv6gn+ZapUERkZ
J3LSOkOkwvqGlgkrTKYN2eKCtMOnZIS9oExQjIaksyf4VW595bmU2Tyu0VMd
hi9PE9dnLfjxp4NQ2M+UC/+UXE7WXbBeKEig4Uj7Yo2xDaNlWqPP2GSAAKkD
bV0KZtM6+amWFafAH10fGVrF+TiGtGduuz7wBwQQDOFu/npnJWMv65bkq08n
Ov7j7djdfROUukxFdRMUK+BTGBKUWIkeCTNeyaolsFy4Fa2cOyhS5FNEnYHo
d92XIDr9De/S0FMnrjGZou01jed5oGQmH6ykvEibqmupsXnLudD8tBczr/ir
l8gf4l/kPR5mVl6X48Zn4/aKbhxZNqb0CzI0ohrcAyvqjrBZI4nsQmbSYMV7
gOiKnUWH2K6KkIQwdSYd8cZPeyhJlvPLQhhtp4FkmZjXWfi9aMZdGeU481d/
J9fzLL8vveUe57bxDuMPJdcnF1PjvraD3SBSRVZNkRUet1UC0iCIJZtc5YbS
Y1UaF05fuHCdyAS6xRCaeBE4lAwQNxQCjhuYFq7YzJ6PI+9eq4sapmsG+jQN
5WxnP3nWG/pd5XJS8OiBohJu8xoyyepxUpgESmrr1pzG6hMZ1vfkC5CRsRkx
ALWjW5jknmuQdUdVYznztX98Wumc0WPWORDPVLtWGDWb/vdmDaW8qEJQEynq
Y90x7lE/wb+oeSdqdKsQ/iK57ylE4luyrBfYwBJ0iZ+1xKEhMO9SVczYPKUV
Vg0h6QzmrWZmuN+bsQCjP2vwR78fZ2cEvlJnyTM39/K6bNRpIqPj4fiU7JHD
1lsnmalvLRf33qfu7OiBzV88wJUTDq2qPifnzuD2NdVO0q4vqibvg0fzGggY
hk/DNHUUmrCgNnYKTtCJzGvdUSjEpwnl9K/pL5CjY8BMMBXHKic63nP1xnZb
3HSn2uZBZ8hT+dGcotPfW184Nkj22yfJh5C1yLuuBrrirly/Bf6qKtCVHvNS
z7eFOFDOQvl9GtPTB3VWVVnW2qJ11U3kZFL3B7vMUJGxYJ0hdtLnNXXRkY8B
PzYIyrU9ueT/2EkuT8MB5rXm/u2dUphj+5TflY5PKVQvKc2/ud/faF+bxW/6
fYTn1s9uowQY2F5KcHUcqbEqbxB2SFWajmWa2Sq3Ig0Br2TLgdKwOBYaVVlk
0NyN21Wq8yOvREPRJFXOgATLbAufoytrc9N62jjv3R+utI9PCDND43jeVWPF
3yt7xrphj0Z1PmvLz+NyVIHSA4HGv7cdO54S7/ze/sgu2e7GI7FhMB9V5DNE
66LKeFq14MVs04katkk1ddUWHdy2Jc+d0S7KQe32y85ZTdvNBhIGTbhWAIcR
dGqUpswn606dm34n5wGSsZwCJukR8MBn6lPVGBXUzGKb+f6NswE19lQGoO40
Mv0ruZCaHLfbNLjTjzxZCeOqwSmdlUMAWHXczrnggMlZzdreIopRqg4ymfVF
U/VVTkuJNW1MUeE1+LFBUVNuifJvXfVnhdpl4caXdjXsGlg2d7OTVtrekTYa
2qSM21KGUl9i9IaFw8iq0DcIpjk6VjMlk8E3lwvoXmSWe49f+PauN3KwllIl
ICOLrm5caV5eZQWsjLFpDbq3tDpl8B2ouSxbQFXTGhCXTUUICDV/VBcrNHiu
xjS5rQrPfZkuS3JnTy0JBBQMy6oW4slgJnkwbA83zpDnqLzBj6Y1b0Xv0s8z
HYYwzgG0mCo45T9qtoweivLx1g7t0Ba3uf9GnZ0GqYhqFNaWdVugjSKvOpPC
hpm4bDJWgT3GiheFtU0gYsikrTPb5WnWNzHtwEdhaAXJwSqmq74GbdmgFQAc
tHHmjnhla0QGHrRAMo0aA1XR6Kw3gYuJSK5N5ooLTK0ucPy8aGdOfU6wTVNM
M5eV/H5y8YMNA+31Be99++CVc1/PVxduyhiYMFGVfVY2FcJGLbLSVg3mm1Qt
fm9sZ+sit2ibsKbskcaurGryukuRB8gqsl8N0pRNnonCdsXgNFBj49JjHVbR
WBg74DRUcXaksIWbyMQINBfN7HZh5qySjA1lUh4bMIRE/qYjU2hoArS6TrJ9
A7k4q7Q9Rsm7JOk+XIU6/QI2CjDKZsJWK5Qr9R01zRVtVadNQdC47qq66bMa
QQzsGkr8k37VpFRhoTub8hy5ZvRhIL5sNOkPGbHcRTnUyoR9y5nhlHBpkD0o
egd7deVLCTPH1/B+Ng5YjXIZ4/l5/szJZahpPqUYRBD8i8Km6KFon8+ovM3N
EoB530mRIcXCbJp1CEpYDRCW1WUHl5JWXV83ZWOLpgMHkzVNU6ZpbeumNm2a
W1PDQpme4efUZm3fljBRKatrViNQyTpRhRWjefyEAWprWiyebTsaVwpGVBNj
SjuO56fuS1/jNNw3XQ5KE1zeZLfkkK1WF8WjyG0qo/6F239+vnImscqSCdF1
2aKEs0rzLM3bDne+KBGrdAxQrS7Drip6k4et7eK0QNzSdm1TpxnB6dy2WV5m
Wpu8qcjQ4XUdjcOuGwSYVBqopGCi6K37KwE8PI1hpjJC/N2GIDSKO+HEKezM
GzGblzHwA1xMLscbMM6ncXU+f0aBK/8FchlrUXpYljxHIrIvrLFlXPa6zdui
j9NWg8svVdaUcdyXwLkszfKUNX2d2rawrWkhzzZLm3JdttUqRs9LVxRMI99s
maoVQDW6YzJbUERpaRdZMJ5NlUH1iC91dMCcmoGeaTprRqog9vEV6rrv3jN4
8H8chvEl+X2idwXSV32bVmlaNWle9SDqCbJCD6o6D5oG22SGZd10ddvGaVn0
fdrnosvrtM4Ay9LWtm25btqyKhHNxPXaApTlpjEGV2hquJ28Q1dMEwPzUe0M
0FcBuiCrZwOAXBRDpR6o5HTZHG28fQLRoM9m/5BCTLmiO6Va8n/p8Xu+XCjp
YgK/U1LVxGXKRZqb1MA0tVjWJsPuMqZKy1ZDi+Di47K2Iboy6qbgBp6mbUp4
lHRXIgWTlB2G0xRVU9bathrwoM6rpu7a1uYVyE9rWY69akTVZ7ppYPCQFSjg
f+C+UDpA5gy0f62BDPKchgcUmBWsh1ZzebnY/LxbXA4L7yN/JW9MXPkBcqk0
ksYa3kM3FaJG3kMYlqwM3EjdYfFSC2DVsnXatolt06KOw7Jv+7KJYwjFprZP
LaxeWBZNo5o+SOlXPIlkQJ/EGRpz27ReoX8CV0J3AKgy3PRgyfoaNBwoA9vE
ddeD9DR1bnUOO4m30w48NDKefopzlV3pO5SXGy/L8+IAqdRlTwn/UXZMdAVy
KSptO1AvWCebGtilXZkAd8Vtmq5Tm6ZxaFla5m2YJmmTJnUGe7YK4zQtsbSW
bFxZh+UaUqihVPgtqfFCnoaA1bhsDa3JunWpCszjzLMe4xW6KkUCM69y3RuM
ijOmR4GTqdDhl8nel8y6zTTQEzMWPfn0jpqNqA3GIvVhjIYr4pAz1DYJSP00
/yIH3q8AQWkRS64628JmpSXWOq2qooU6wHqViFIqSKpNyjRp46bu4XHipK0A
B8oybvowrUIQ/hBJCQSA/9N2bdpwnaR2vW7X0DfyKn1vW1ipGi8Amm7d32g7
WFE8aIsaQSuAR1GhbABBZ8W7SgvqqiWmiFmiSylQAcYW2YQB/IiNofWTTS3U
MxTNIfbuK8fGfBFOZrYgk65hfOqM67QFQRk2FhArxxqvwhaqUJbrMC2hClCd
uMT/DX5fxXGdxAk0pw7p0TUcTQtTFzdJWCZpaOoQ2mVtCBcTQ5VybIYGUjrF
72ltkBJt67qE7Ou2T3NdN+2qETCKaV0hDjKgrW2KsDR3gStVAiBnSk3RtSsc
pD5EF5k6soK2k0UGhuuTXAZuRikqEtU/r94SZqqGA8EnTDOT4sYPy2/DnYUA
ABPbSURBVJy16xg3fRuvCSCHqyYO25iOFu5/DYVKIbY2Xq3J3YfwL3ESx6ty
tcqqNR6AwFZh2NB103UcJra2va3CtiT96su4r92zpFrQSYugKW1LxEwC6gE1
guUrNTaytVncMvgq2EGXg0BnDgNxjVJDUiJw32SA0X7miqtt3WZ8rGQmmkyM
sxy57w34aXJRCJN12LO6bVgFB9JgHcsM6AsRJqxWGTZ1nKxXcZms0iwNQ+hC
mJRtmWD5wzUUJQ2xqmGIX7HQZYKH1mAEcPqKBAkhhhAgniPLFyIqxSVwekpK
2ONEmMwWcBoiLNusSdNwhX+arMugUE1uitRSd4F2gK3tYAl7+Cw+7QOIySeF
GSqXELSOY7loxrCHD5bPdqMvzPku57UM5nnPWe2W/NdyEZilhpgBThjEZIMY
EcRJk2PZwrbHbZ+WWFMSRhyvSXXwX7hah/E6idf4YQ05JHhwjS8nqYQeLaGA
kFRJagMNoxfgh3VatwAOdQjvA5ml8FcreLMM4irLkGQFqgFGswZwy5Ggydqm
qCtbpkJbQAjBU4wT1kRcE0MnGoFUaA3+AGxC70YKoOiABtsNxVHwK9ZNRRkh
g89Fm+A09E4NrSLDmJVAXsuZ/gO50NSJwmRthRA+A+Lt0yZuchLJbh2nPVxM
nKwggXAdrxKsb5pgsUMnjzUeWq+9PEg6K/yGtSf3E68hojhJEogkSdb4lU7E
KVChXbki+Thtq3EF+K2Q4qCmhedCfARojidsX9ZVAWYuRnyU9sxUdY+qdQWy
IAUwQEjTVgq1ORk4VLA4LSKftoPDwThOJAOUcXQ//E7HejS76bHqWoImDSzk
ld1EQlq+bzDTV+gL3jUiyhZRe1mvYETisDYwMABgWL6E7BLEQDqAL8gAbqQk
jXGiCWm5yThBOiSWhKQEiwfxwHuT/uBYu8ed2KBMK/wVPBu6F8PhJA5R4KLQ
lgRCSkPbw5c18QowYI0fCyaA4HJAt8pYC5QHHsg2znWVKDaskXZgqRAow8nq
lEQiKs+oZabxJaMFJXwcHqD5XDBdLL82rHhmtzL27/0L7ZIIrKNbMFq4W2Fn
wLTAiyfOeCFMgfkiL4LfaYWTeBW7Zcaqw5CRvpDarMm20b8JTiEdScjQQRbO
xDl9WUGogM0k6JIkTLYQapMCLkBQpGe4RtivS1hOMmyt+wG3SxaSl6qA0POk
hdgqTXEuAAScUI7+AiMa5XLTrDUZ0hHQrpzaatCQrmmkSk8Y2vt9pCYsq8cW
z8sxaGJ0LOKdxbVfIBeBgAElr1mdIUjXFh45bcifwPaQpiCujxNSC1IBWlla
cbJhzmr5FYck/O9OaVYOCoT+MWfuhifxHa9MSnc5Z9bI/EFSoTODTs7480m5
I+uWkmMDwIvbFSAgLCNwSElBbp2UGgYW5yEhDUMLJ9W5+QOtQS4I9I5Fcq7J
MGioIGYvAP8KkQFS26oaGjucgHxRG8IaMxRiY0zRaQin4v9cX/KqYoCiIXhH
YC+4ETAnIS0pVmoF5XHewbkL5/D9KrsvWlBvz+h8LwNCAbTgq9CLL/TmzR8h
Ga/EXZek5CwjXBdkuCLHTwYzQZxKjgsIsIWKQjvpJiGVajvCem3SwI4Cpycl
OcAwadNVJVWDZEMP/qet8Y23qc1FCqeFULSyVZsCT0s0V2lUY5GHoX4eTxBQ
F27nCtMIhRs1q3aSj3PRX4LHEHIVeauAimpCTiWhK4giJm+O+KSke9r7dL/M
J9HQuhIqIHE5FLYiz07L7fSKzvLyW3tL5v8JSQr0shVdgQRLiuT+KAwdaQxd
BkatLp3AoWDhuoU21esK5A+cD71LXAosEGJagLg4FYDY7aouWZqXDjCSD7IW
PVEQFzJ6msrauRRIKgng6BwPtPrUf4PONdfCjsHcoEuVK5tW/0xfaGZoUaEd
UvSm4qZvgwyRNsWJq/XoFb7H4YF5uO6zdj1YSsCNkCRDERM0egXPuCJxIU2H
ACnrS9ZUDSuqCuk8gzQpKAQcXZaCR9AADJxSQTFz5XI0v6tJYe04XoAnYNoZ
myo7r2yOc9kQ+lS5KJq3iz7wuIVDWdEditg8haNHPOlxcPJd5LJy+ghtSskK
xgTgVhRP4RcXy1Jsi3dOZhLu0bFEO/JGlIgAFZs2PWuAxlcVIc8WRTkdchlI
NhBRrWs3Hxg1CLZrBYYFGx/F8hORc6pOIQekUB8kAjXH0dGzO//cNrlGIHkP
+74KvQd2zgE3n/cQ3+Qgq0hIj8yge3/OZHpQTr/A4yCyIjHF634VZpCGNQ0x
TGmYNMz0ArZAQBQtRtynrELCtGostnhioRtLBEo1ccNsQGwPDSYqExch5rBD
ihD/D28pcYtgMBUFDStiH1cDuvo+chmgBoERck0OWHi5gIhzOA+a4ikgfI/p
/xWANjABGAUix8GT7iodZ3HSiBZUhu4RnyLPAO4J+W43bUVXVAIqdXHatoF3
LHiVkuOvd398olz8ZUVrRAi+GEkwxGpglVHt3ROhtR5M2XeRS7gaoTjRDLHH
e0AXhAm8XBAYIRTCQ6C5EX1RSJSEGZwPEGWJap0QtVZlnfSU1+tbJF1BbVRI
rKbg0YkZUJjm7eLNfFZPaNj1Ic4XXue5+kI4EGPEdQ/QBZwc0qehz4WQEjck
Ar3vJJjhu2dz0nggG9wDDqQ7BEdCacGuxsjNIfpJY+AEkHpAbNZ0FVUmoFqk
hUPNmrYnPFAC97jaeeHHDcthkiqVAvMx9SnvDnD+orxYQRMrRIf7qiTeEYGe
c7Tfx++fpOONWexBmre4Mbl9orAJIKfEEsUrmCqwEg1Yb4AEIAJbliCaUP5R
q4asG1KAFrXYGWEvl3jLxsKoaUfILrgynGbcLlDKL8RjGE6hqJkVZCvuEZQ/
WOwDCoa1zlfgnpJktfpG2rLyBMNAK1DIFIaej4NDcRRCSai5yyAbwtBxugYb
kMI810hjl3D/cZbHBoisByNtWw76hvueQzm7QUV1USrw0ODz6MFxVcFsX5gr
9QaDb0GfBLJSLa/IVZKarB0pglgyJTYMqSoXn4c+FBx+HNzwajQsyXpcq1m8
40xi4sxO6KgWR97EngFd0QPDz85wDjDYXyt0dgnPk1I4R+HslH8NIvwwSXdh
uCt3LW6cNYGx1vEDbdMmME0ZMqREaNqQCIK0NYDIaDMskD8oJaPORHCBeYPS
T0rdUI6g7rjfFQ0dhDn/wrrxs7zPzbhyarxiKAMH7HdeFB9/5Vh6Rxmv3L2I
m9FxLoR3TnjAoSOSCt29q+m2dmF8nHirE7or+fjc2UfntWJPsjmprd2qh+7i
dPeTSQpdegd5hdhdml4JRiYh7jpEQQdSOBbMJQhV8N5IXReaIV/UpyieLtfI
BxQxSGdb1lQZ3eQMPkWgecDNstEeAbv+dE8y46Nb3/skTs2EX1E3ruZDb6J7
IlSnwlGuadSIy2lQ7gRLmniKhTj+ZFj8hO7vk84kyTwATSYeLBkiDPqRpEAv
p9XerUa62dEwDl0lnsMZZOPDkR0l1JLE8dorF4LAi9Rr1OfEaC3oqz5F+SE4
l4T6cEyG/gDUhqLavafyKUapaKT6Gi0kci9U66TQPFUhtew0wrj+Pz0fEDTS
YvxL9EVNs3anV8hbE5SorwGlwug+qojOSBy/DmIfNqx35FJWdEVFNS60qomz
SKHLHXuZ+DX0rP/KM17ugKEhDOv4/xWlNWnld6SJCVGQnp6mXI4LZEN3ddz7
iQveoQlpkyAiRB0AMpko6bQo6Mj7vsmIiERAiGqQsqDeQlQNcvSwZ9SLlaNk
hOXj/QjE79uV2LAcUAk9DvByI4VkNXcgw1NSfmh6xn2/Txdvt9FuvPj2wrhd
BkkYwN8DjKBCKfE3NxKUZEF2VGBEi7ZysYxHZ1jgmLLwRAXQEdOdvh6VZn0i
9h1mJXiUuLWHYpQ7kodzE8mOGGJcnYQbUmnaDnVqxP/gQWSTU7Q2QQIpvSmN
YowKhbPojAK3hS5D7fdBGerHyDRN07mnZVfq4r73k2aV6M9UYjYzhSo01Wg0
VPA1qFdvrd5mb7+CChPkxWZfc4HpvO9Q1w3bXMfEJq+dryEdSVwyzGFV+sE5
hNVE/3v2ZnQw6yHLEpa7lDhiyAH6QNKIS0f6IONYUZEa+pWpggmUgypQQdYb
7hovUA2LYBvzGMBE0FAt3Q379w51YlOi5Lz7MusenWgkL2zHUGrxoalN0QPa
EmS0eXUpx37kC6GgF8HtYOSm6FBxdofOCfR85QUo8QzJigrWAexSX7v0Evol
Ycasa9hHwimjtiRYOBxVDytP1ZUoN0sdqUvpYDrAIyaI5xKUmCFg7VGIhKct
ujZyRNhlD6MIiSDWbiuUMfeuIsL3fWWM1xV6AboRlAgrmOvA1Pd3IucjSeIj
EH53AxMZnNuwIHhwN60P8vzVAZ3IbjZ8utm/nD9HoyUxtc19QpTc2w74p6cS
sZSAF+AP+QMfD/gsiKcKnYMIXTJr4A9dQpKSI+VIH8Yrn0lxwqk1AjzbwUfE
XKGQP81rGBHUUlAfDHrKTQq7gp5y7VZdnlwjhghkvquZ+FwE4363jIrfn7Wm
vFzUeD11fccxeX0Hb3Wq0eRfJpegOXw4El2OL9KXHLFkkqpFLt9MLmJr2dYO
+rrI5XvIhbxWut3sRkO5yOU7yIWfhMPv8zDL8Y/mji768i37kdWiL9/GjsmL
wX2LXH7yHPjff8gvPS96L9cpjhgWMz82+Nq+RHeP33KeP162w0u+6Lwje+9d
co3t4fvHXv1bzsPuePyrz/tsCzq/sz/sbzsveDub/j+cd+PtXyP8tg9+7l9w
XjBloR5Zx2ef987tFNq/ct54tF993nO211Z/7Lwv/8PqCxHi7z/4PzrvASnL
jcMw7THatLekjOe98d5tolutu+N59mVzmt14Da/vN8f8geuN5925Xr7dRC/p
/etN5917f+kxOrhpwfsoupmJjjebA4HiBkPdm6cIxaXzwsOW5NPu7py8MR5h
1wdtb4DH6Tx7xwXvGz897M71pvPuXQ/5/k1x//1N5925njlycaBlTnesPtyw
PQaz3Pe1O/Ep+jLUlBX7yulLs7uTyD4OOxTsSvzM7p+XHQJxQ/9Y5PdLune9
6bw718Mdlm0fuN503p3rdUeu3HJjji47BrdyjoJvrZMLf9KMJvIzB5vtB7Xd
31RXjCyPSXJ7VEVs8/vn1S/R4YY9KY677YHKGe9cbzrvzvXoozgzdu/9jefd
u9765cWpyQbKetQ3PMru5XgYPMG+eIq6kBXZIfnvkjOCt9tbaqhVsac293uf
eziPNj3qo7cvaI+5XoX3rzeed+96NOzePCQXf96965mDUTCh6p5cguJgNEka
zXyq3QfPwSI8ftkcX4aSv5vq6m605r6dmM6jIqHj22otXqACtIXDnetN5925
Hm6xrQweeX/DeXeuR9Y9TfhdO6boxDIZRsq8fN6GqaE+VmUHqpxmSvW3mGaU
lmmnL3f86nQebjMbsRtMVs7K5L6fns67cz0PEMR9vz+ed+d6dsuCXXzf7wfV
tlPuREyz7aOnOBdfUGX3waondAntfzuyMdFmEwaoXL6DQ6fz0s1xW934+9l2
sxdpehcnj+fdu56AscEshfu4ezjv3vXWwMlqjeH5+010020km2gfJDX+7mZb
BMHPCd3EM8Nadbbh6e3bTgUqWI6bPTXqefT3gxsh8IXVeEgyz6SC1MNs06Iw
76OM5CdEwp9KYAZ/NYuuXo+IEv8H982Df7+76Pc+1FPvZfnwFdSiNreFooOn
KYx03ajB/8y7/1LRnMfLqj2+mE/ImR/qh85TtGnjctyoK3w5T64WZthP8zzX
+gpLq8t1HhRm1891Tr7WzxGa2/2nndmv1hfIJTliB02x2+wj2s73iMVaHSgT
50mP6BDthy0r62izOwh12OD8dL+nPoRmvwG3etgfd15mh/3+WLEDyPWBoBQH
Skml0SYJJP5Wk4jDbotH9i9Rag8vt+nlP3y8BP026Dam34siaockTRIJQQQj
QFN1LMTe+lgeNAkydeA1uq0uIyM2hTl0QbhmWz024O13nTkK5KGKgY/a7Wgo
Z2SCbU26We+Dw4GhqbQ5KLXGto1sEcEbctlBGoeevoOUwr7AaHpcJTIgT6GI
+/N0YkB0GA/6fbBBOvqYE1cbFfZlu9nszPbQ+w0QggNYusiwDd+l/voRmcX+
QCSjwqXtQRLz/oKkMbX+xNkigQfkQk6b9CQM6SGHevM9+P7WOYoMP+J+9yl1
Sm7v+9pnFll1GFK/JA7YpkO9Ya/kQvoS7yRSDfwIsp720A6jZhHBVQD1oiY7
lm/gtNm2I95XsMH/U6591/hm+WMB/wHLhP0rXXJ7m5sNZlLTnrzdkC/ZIxUF
c5dvxjTUzI6pbcHwelJEry/4C7t2kcK1QyOX4f3+YXM45jyNXjYHjoKPYzv2
S0snFzqaaHuAggAU7BXJ5dCT399UZv+yH+pXkv32WJM1zAaMNvP74O+3B+hL
74xlFKXr6HjoFhlchcpmmpMAb54P7jspr5IA6M/fhTNkPO04JacSXuX/1d6M
ydsxzBJd3j9QCxa148CWsLzYfM3/g3zVTt/jT5Q7cXEbT/Q1Z1yM6jbAXXtX
1sBgjKJNdSsMFOUmOm5LNslRINrZRGu9rOznKX8/ztYPxDtLksjTOfx+Fa9a
KP0nSkVdEPHBzAPIm5TzbGvDRRBPlcs4eE6qO3kx/r4NwBe3/rwkjLo/YvBu
4uXN4XbL8W65KOde5J28MlfBnTYfHixJ46/QGX62X4c8W/lbI1jfcDDyu0rm
PwKn1qPUrRbdAAAAAElFTkSuQmCC
"" alt="Scatter-genesxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-genes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 4</strong>:</span> Scatter - mito x genes (Raw)</figcaption></figure>
</li>
<li>In <code style="color: inherit">Scatter - mito x genes</code> you can see how cells with <code style="color: inherit">log1p_n_genes_by_counts</code> up to around, perhaps, <code style="color: inherit">5.7</code> (around 300 genes) often have high <code style="color: inherit">pct_counts_mito</code>.
<ul>
<li>You can plot this as just <code style="color: inherit">n_counts</code> and see this same trend at around 300 genes, but with this data the log format is clearer so that’s how we’re presenting it.</li>
<li>You could also use the violin plots to come up with the threshold, and thus also take batch into account. It’s good to look at the violins as well, because you don’t want to accidentally cut out an entire sample (i.e. N703 and N707).</li>
<li>Some bioinformaticians would recommend filtering each sample individually, but this is difficult in larger scale and in this case (you’re welcome to give it a go! You’d have to filter separately and then concatenate), it won’t make a notable difference in the final interpretation.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-umis"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: UMIs</div>
<p>What threshold should you set for <code style="color: inherit">log1p_total_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-4"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-4" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>As before, any plot with <code style="color: inherit">log1p_total_counts</code> will do! Again, we’ll use a scatterplot here, but you can use a violin plot if you wish!
<figure id="figure-5" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAYFBMVEX////1
9fV9fX3v7+/5+fn8/PyCgoL+/v6AgIAzMzPp6enh4eGKioqFhYXa2tqzs7O8
vLzFxcXU1NSrq6vMzMyRkZGjo6OXl5ednZ11dXU+Pj4fHx9hYWEqKipPT08I
CAhxVciEAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAgAElEQVR42u1dB2LjOAwU
RdK0SfUut/z/lzcgVW05cUkuZaW7TXHsFEJogwHgeQ9cymPeen3xJTz56Evk
emr/x/XwMSuxHtrXq0v/5rGXrNdXX4/e/Wp1Lj/QxazK8n8JRSK+eiQWM+uZ
/Q8Xe1QD+O60W68vvw5sOSKT7lHV/Ztcuw9VSqn1fn/12t20Vzhf+3bmV9Qq
l++Vi9UZqSAP5T2mL2qVy1fKhVRFdhaNSXqr3Ce79dC+Uy7SWrHefgk5IAOr
XL5XLl24RqIRzqrd6fetLVtP9uvkojyp94f91tsedru8F9XHclFSrtDmF8pF
esKkntknQTDBN3cfen0pxQoLfJFcpHMxEI23T/xgjJXZ/o7yTKbXk/2qONn5
fe7FOxMeTkdDn4rmdHzzPq4acJKp10d06/WJ+uIiMOVl58rT3GuPXWD2kb7I
IRldncyX5ZW4888b+7E59Mqwex99tn6JXim0Xr3MF+Ewnji2+AQHHPT6Ij/U
l+7/VS5fpS/QluwNIXJ13O3OmT3zj/P9EVJb7djX+H3ZhWBqLAXQiZ+89zkB
BHeq/gmraL5AX1SHiPVicRjmh7hlJ9JRcdbrs+3Y57xivVa5rHJZr1Uuq1xW
uaxyWa9VLqtc1muVyyqXVS6rXNZrlcsql/Va5bLKZb1Wuaxy+TK5qK7ctlb+
f4pcXPeFoqL/yl7+OXIBO1ZdEjPW60f5l1UqP04uUrKxb3O9foh/UeD5R2x1
Lz/Lv7gGDCNWWtnPs2MuHFvl8nIfn21Omj92WseRfHs/sloIotZ8/5v5/IMs
5EyBVrl8t1xGYYjJ5MTdGk59r9+HjxZ6f9q3nj4fjkXfN7Ff04/vlQvBizpi
5py2pWw3q3/5KX6/m3OxS3eZZ3Zdpv5xP/J6fXX/i30T7WxvpWtDbk7nt/XQ
vj2vhIKYfW5l8tanLY/aMTQzv6uR3tLQplUuH+gLP/pIJSPPnOwDD89TEsJw
b2FUo3RdZaqbDpSskniof59RPzJrSy/wPfmc32fvDdC0HX+klaskHuxHPu0O
abQ/nBEn0xE/Lhe5DIcx8fzA83X+GO9zS5wee0Zf5A2YkhntmpbdfKDVvzyg
L3aEL+vvZ+XdOd/y/nDAVWJWdXlALqKrKo7YmHohr/xgxAJfZfPAXIW5AVJP
5vvqY68jmnUs1tfWxdQHVbEhPp4JSPB15P9XyuWugTD2SZNnklYysfi0f5kI
8GlymRdpFrVFkap0leR5BrOuAPpKfWH3naJQU9xFSakWBSL/aXDms+SC+I1H
5rZkbIbSz4lVox4oN9ZcLc7X/IcnZf1f+gKZqSE00PIajFkInv/l1Oaz5NKN
5r15kHycjS1YwT8M4yTL2L9cF919ZiXtnQBKXarCBO9Zvsw/TYfaLe6mFN/X
l6RWavniKbO1X+wnykV6+rjfHfUql5/mX85hEYXn75KLXHtkbujLbiAhf5tc
1CqXa/+yR6292H2fHROrVBZPOd0fDrvi++IxzVa5LJ2ykVww821yYZFe5bJw
ymr3AUlsd0kDmEH8qiM2u4Wl6gNXIvSz5YJ/TS5RfsjzvH7EvyhXUunPU8kB
jRQf+HfQnxcBznWR/NUpx8e34/nox3fri5CTYfGOc+YEAkFF8gNcRrJlga0a
c60YyQfn8kHnHx8hYpaJZ8gYbK0qX59yKdoj/Xe/f7E7eHjfwMRTbU2YxTA/
OGC1EBELxaKIrYbsyo55edPUeXXvKxwDzNi4WjGdelz1bJePnP7NiGzVlyWr
9JGB3y29QPTMFtXbJ/Whm1isILsXryHZQl55Pu1Oj+Aw/eEaJmDE+hhAKrm6
78+UyyHJjM7ufoXqCYC0T6yrJIteXcSHVqzzRPI2e/yhcTLy7+JjRzPbj3x1
tKdbtUh2qSBLCiNcoHAxokSKWN6EkYW8a5LG39YX4TVtg8zSunF0vhxOEuuR
T6dETF4hb93UaqRfWHMm5R33tRvo4y1zyO/fU6b+ev1lfz4e+zg5TvCENnhf
XyBB07NcGZcAuNwh8YX2F6nmiuHG+HmO8reoLfeFAdLBDn/XjvEzn0ZZGmII
gulCvv3Fnaks2NhhL+AjCcPsUQqdMSmzjF+d6zTL7+TisVtnKu+nBUj1p/Ul
kv2ad1wZepHbw/5snB+vrvuR5cgKkD1jr0O/+g/NlJgkpY64nC/9e+f8Aerw
e/4Qwf4UfrMQj6H8cnJOBB6X9IULXh7lUDe7CAXUeLP35NWuQ2OwTBcw2Oyg
pXNNuhAUlC2hNSa6I9GUxrC/LRcTGVzWOxD06J6QnQafcM1s4plrOVYjd1wO
hhCIv5kdmJwF0PD4qlMudcOOybvyf8bUX2oA2C34WWfDTvYwo732Mk+UR9Y3
qO6uqyjoX5FdxiLVJIaSnaBvnDcpmkzYrH1fdaEdS8VV1P0vUcl3SxGQtS0U
d+nj4W0fHncnWo8sFvsrbblEXp6c05dIv+eolaLe/usePmkr/ObC6bB/iyaz
WOWyucduDJPkwMhfyPflGM9aZRvPEsdeiPcCKGFnYYipHig5NI/PGmWolvMv
dZQv8S2dJE4XiccNPNkN9XM+RRRmNlCOyAK3Yq0ujZRKeN5FP4wtE5CEJuoh
/nG5DPe3OOvLGQhqqf7SJ4VWKLzv/J/4/nci4B5LuzRXY+ig7ufn/gP1yryB
dY+u8oib9ReXHDI2VaxRqpH+sBYzWrtEj6jnAsAm/107dmbp/ni+jU/tPmgp
p3FyONqkozoxrnXK3svT5fT4eReTkX2bm1H5T/H9rk95J5vtHAST771CXuDw
lGYaM77EoL9Iv1dHvlAKJVRfYJZrX9IUhykxnu9hfnLEe+kwAgiGpRVqKf9Q
g/WSF+2Wc0XyvFUuo4UPai8LHpWLZn1aOOpKLwbBR7yti6P5DKdZlIBc65Wz
q6ETDp7oR+7azSIxCW5t3ZPeutu/b5nk6XD2fLk/TfV8wVUunZ/eDc0Wz/GT
2VWHchR1ysOMzVqljda6Q8/MjT7ada7C9KrAt9wf9/sXeeOyH5PlYYEFdy7E
ymWkWlq/f5ObITO27Hf+UbmY6hinSSqfnbjsFENoiMPhaRmfuPEeoxy6wm4G
XMJ4nlzl4s2auPmrfa/STUgSo1kbwS4AN05+uvsCNws/L428sXKwygXnmBze
Dm9vT0z0H4CyXglEkl3h9J0vp1GjhQMnRZrxJS8l/+kuy4V6ZfbgK8iNJ/TP
eYQJCDnD+aWzS3LgXthhigMGozrgRU0KMTqlPhr1L/r/67rYmT3Igx25ZvYN
u6YNyVGPXJzVkcrhhmb4pxyn9diYAGGbnVkiVn1Bvt/WeR4/jEDLTjImHk/R
FPnY9y2GZ9rTFxHzlvJ9OXQnuY/0vzkA41ouLdHHjg8zze3trS6H7zA5HSza
OXzHFx+9Pese9fovdwZPDZN8Vzv28RHsFvj3XfQLYjMS+dRMv4dG8cYCyoIN
2IzqrZprAyiivkI58S9ePwxT/JNI2ZVc9OP6wkYWmRo8vXEf8bjQNr0XUcGu
2sQopUncYgVn5mQR8fndIdUaj9krzuPyfrnQTPKUyzGpJGiFvJPWjjfGhqIa
m/Aq+7IOIE02TOeD+GRX458VxsRqxzqDwu/HYWZgV3cVg9+fxmYUWEWuRKZ4
1CEAuQPJtG0tn4QMrmppY2ulVjvWHXX2/HwYe7RLmaKXcGSarCPh6ek0vg4e
sKpja9Jq7Re7chaU7p/y5zf5iEi5BvGBDasuMxqTORGsc5Me8+LiFXxM6aKj
K7k4wtoyM0Pzmb7AzOZcnLW7cvGU43Ybv4Jb2hR9LE7yiSRmjshNuB7KNXry
BalWuVxex33TnNvX8OSObwYujBUEpymiU0STZaoHXQayvuuZVeNkk1UuswtE
cY+/Nn/MVKIr7GuBnFGkQhSRG35h7OxkmQ2sVyPkBMZh3jqv70a/2A5yYS/O
6+ODHWOFzVyUHRsDYxWbC96Z6A0YkwmfVAJWuVxc1eF4POQv+H1ppo1g/AbP
VnWWDPxZpzAx4x2mJlZ9WTplE+fDTX08QHNoSbIeqsG7SwxNXS2L83ryCzIT
yljkyEdyzfrIXLN0EggUE4EZfc94H3XZg6Rubpib3hTq98olJ8PT60uaUj9y
rRt/nBu3SMCTinhjko+C0qme6QtLi4kUk2mQzMZOPdk1ql3p1+0S6X07gX5Z
FedaLic5bQbnB1vBpPbX7pT3S637wLaqgqCyASVjaapHapltPSvGzmE+O0zj
TdhNcja939ygNi+t8vHe2+vHrlc1/TJ+Mv2Jg1wIkTmofj2yyE/Ht+v2evuR
u+fpra1CSkvMZ+MYv+mmStblmcLCmLLrXhK9IYvm/NrFkE+zh6oTMhO/vI58
3KSZP+DJBk+Arsi3AQy47BdjM4I3baHg3ZkxlcVqmT/BHQajk0Ep5EQyE7mw
G+VKzS4WXUFV5O2BNL9fLp729/t2AGNol/hb1AGZcvAvY3lEg2s2hAkWl6Sm
byrzR6lxEYDolWYyTHw29EXbJEZ7LwxRZJGRH9BAf7cd66sjrHVb3tE2toXf
HwGA/cXJETeCUZGyR4Q5vdNplonOa4gLSgt9lER2s+jYnClYxs3dW3y9q3VK
nL+7YOn3+/3+OHb0Bv3Iu9rsdzs93yuqLjnjkwkW2ayVWJIl47ZWPNEGNWUx
2epXAc9EVs2kM0loeR9P+a4A+FfHyeN8HnlnZWAsZzFsCM14wi828ABTng3V
SYcnxPOuMNXTAFx7jLBP7euW6hGw2f684WcuDlwQ0Q/2ObtboOPN1qTdVZu4
GCsEWsoUYIuz+IiL4z7ymp3MCPNfRcEmZbPBDGOjuho2LBpzx33vpjQy+UsH
ad7UF3G+ofW75dVhBBIzm6TAUZD1KbJEM20yEk9+NbstM+BhCDNjVhgzzse2
BQCp8l6iYrqhxvDkvr8O8wPeH4Emf1FfkgXJuvf3vCLr5r8QEZyhKMaNoNQQ
qoO737CikPOJijJyfkMwbiRLu0qyNXe9fSNuGYqeCc+VbQm0srFjNdRktNkH
jR4kESZulnKU+skVuOu7/0hnsr/9p+8W7Z52YCMv7Age9+dm8OV2PiKLeEdS
8jIhCOS3cVuBdhgVuSEZNA9QaR5JkblsUnppzygfB8yOPJl7m3DeqbD9aGL6
1fzk7Vu7PbanR/gwJJpUT/kVFDVDVzyglhUJwfA+VYwsoNlpicNOsrxIOn2i
KlkWLRHXrbMQ91seMZ33+wvnLF3KpSh3ZVPH+pG51vwqZC1Sm5GzgjbAUpAl
00kSg2E9U2qTqQo+BLFscvzCm1GaaZ/sIyGZWoQ35+7lF/kXPd3cKu+xfLFF
9NNovFcJiCTXXMRo37dEZD2eE0/SVAzjmfqgoA+hzRSFEc6fSDkZwCDv6ruQ
Qw+UWt7A/MOHYS7UkalhbP/A/GRnvIpJzytrSBQhHoTJqnsy2YjCsI5GFs3u
WZPgS1kmi/nxMTc/U02p/8r763swlvHkh+YqSBruxpAAJt2dXqS2HYxBMxBD
Rzb17LsmKssgL4xt3ytonn+X0SfJ6HX6K+e8su0vlg8w2/2zYLSY/Nu8C2SF
8d17EkVXAxMGpDHEUjl3kTMH2C8bnXcYGVeT7bp4fuRFSaokjB11yNqAI+2+
OPb/G6y2UPMJ2fKWE6dpjkYK9YfnXQAN2yWPxGNZRhgk7n1RcRlrSc0WLNO2
H2bkulKP0pjF2KoZ9pixkesnhwbxThLpdGS2wCAfId+b6cvU3+Zbtuf36GML
+8WE5Ycz9y9Ki6KmFIhmZXkVhcVMdvXhftLlBAZDUO34F6rjEhYDzCA63RL9
iKWhM9d42TJ6JOUf7uPbxLG/f2xSKSFjuY2tE1MkEdoAdczcQRqe0UwxKwbX
n8ym5ZeBPg6TZqWWTUAUig9koWmkH+TfjcDiBGgWvx7If4rXpx+bd2ENC5kk
kbE8k5WhuDjiAfBFrB6hnDKyjJciYd5VbjhW6hs2xU1zWx8YgjiMLreaw1J+
O49Uf1hfjhmGVJwfrKTZeSNphnhWONfUGJP37Jbu5oZCFZHBl7kjxcp0ANbc
UVdR13RhGCAZPcFKrCHUnvcuNCb/dJzMdm8HtFqc9vIxBBp+3lBJufAs2CuI
h0aJSmP6Gx7YVwUcE4+bqc7AryQ2M/W6LmiDNk0vy60dvCoOq8vtC1xeDpaT
f1NfMswbj4wu7vT7LveLeV8OLmLeDwgXSZrHbhgpd8WAbMD1scRH9fiCzKLJ
eBmyU1nn2VkhL4oJySCpDmMu5oOW5R/p1Ng9PJFtt3QHs7HbpYjHhgvbvh9Q
j3JiScuJTWeo6UJrMf9hpncsEeowPIlSoUa43qM9GZPxV/PC1tX83j8wD/6W
XG6X1RdewSMXfKEMVTTUYUxk1gzd4cJy+gn8xwaS2GYkhcnSNKvmU01pSgx3
pQFhaUuTPU3ZoDMusxFDVyzSTkHFZv4HO5l2D6/rusKTdc/pswXkOCKbBblE
xGGCLTNJLnKCBNKsAPGZhCWiCfpFLBhWdHd56gYEi9mQK23cSplILpa1xAz9
/iMtTbvXXwFnMLUuGTw9QuNNYheVGMTJccm5zqvCIiyZy1V6QlllBy+qyPp7
+20QUk/5SoDZtEkd5qwzraYkdM8N851NkFF/o39m9/B2u93i3FY1MmB0biT1
vlqaDPxKpaskilIXADg10IkciMyScTACPMxfcDxzWMEJocabLH+RaTZwK8UM
QZ5nlH8BwNx93I53zyuoBRwYZQxTk0WoQNqw175Fx34a6ULXsZ0ZW5sKjbGI
btWwi0GnBoJA1pJ6VLhBhsOb5aiqW1jGbQymFsF9O76Zr3asBzsjtHvhtBKw
lHIEy8CJa9bN7BM6j7UA5p/BEYC3BGYGT2x5mehjyCm50bZOxm1/BwcM2vkf
l9aw68FbcsrUHMXnUhfB+RAmRtnYwUE1sou1Dn9eLl5a637pq1dUxB0XyAyJ
oJyYPHGoc1ZQxpnAoiVaK+N4mYQ7A/RnXWDH85j6mqQVjTtAO07WVaNt8G3E
uNsPtP7pwPO50+8GNAj5C2Pnu+Wi3sNh0HKE0wLQjsEVeeFlFUpfuJVjoonl
gMUyWKcMIQAaZCCcnNdc6iQubSxGzf6W3czdOF+3E0vHl1RYIGOxdpNm1MhX
4mLGmBn2All+zjC4QYylZCl/Rdr5CXIhV0K5IxGOU1idqGRenVK10dAMH1gl
KIhJogJsl4TH0uuTHZRtMnLmZly9Q2KidAeJ6HCQ6XxQWT9f1u2ClxM9mdYz
Mz3n18rOkv0pfRnKUWyhLqYs+E4jxfPaEVTTJhHQgMISw5G78wg3ehRVAPwN
7bpQiYV7YtunnGqAnZM2jAJEjeUxGM7R8HSyzqRjZUg12zY3bgCEpVOjLnUG
71fskb9r6m4/pUrd2P9i4yqB5LsKKLnkmavFeMQij0hrUi7SFGykjCBndFRQ
BumYSHHuZikhA41cjkl1zMR4006lDm7Juj40O8OpYG677NKuWEEFuAl+KXWm
huCNUZaq/oS+CGy0pM2iyvVfXDY1IueA1+dtTohY4qXUzszdP0BdpfUUOcIw
rWtd1gWYsbXNF3MTc64EoS7gxopIi6ThgNdw0EOLrANEeyDUpTiiM2dSmGyC
HPFxShDj873NoutPs2U4zuVf8fsn7WYjqIVXENWVp1v4A0pbihLSSZDkBzF8
BKooddaUtPYzi3QcR3imBlOm5g7kRB5PGTtCBa8O8jwBXBklRA6w511VPe8p
dydOhvJiOrwcfD4fV50s58Rkwi7R0l9uxw4jMsL21/qCtNDYJLIKOJOKe3mJ
yjHKlmXtNRtEY6mEGymLFF8tWWldOp2iTLsMpILBK7MC7H9tJzEWnclR/czy
xFolG02nF2HacOxcLG5/V9OVT+mUR65+tVzo5A/7fW2FUl32I9s/t8L5ezIH
it/QmQclwmUuorISJq+jSmeo6GRFmuasILw+q20ag6QvQUKaoAlG5mgE4HiW
MHGsEcKlHTM2gaSI5cxd3yYdpunKAkl/7w88jetYoaNaushAXG+eV79aLjTC
JTtbap4QbKn0nwZNATylAkE/RVKJsqSnsd6nCGLYs7ioS52nhGAmuS2XkamK
kzQ2tWmApWX4QoOgOYkRMqA8UCVEeUEeww1kyQlhBv28Q8KQKlGjBixePi4x
leOQjJm01Mic5e4v+S0Vgd1dqQv+sGMjev9y/afpLA8SskUlil/lBtTLIBNV
A2p+KvMaBJltngFU2TYV8GSSSFFn8PqRCeLCq2KQaGNqvojtrA3cAIjRUMXR
eSFNEaVA/6MYGLSwo2UR0RGxg8rUo30CJGaxNqUyPdsU2MXOlg9qJ6KobogJ
+9kKs7snn4TN1+e8W+qyv9GFXZXAHQtEX7oKKlZUBYvyBKqAsv8mSEpAkVFu
kqRAK1KeBjFrKnjxPG3ShBYp2hgYj1BN3yu0dVvgCOI7VrFOaeWFtpUZiR2x
GcnH66suEZsRybHBNxVXa+iznq7R73Xk9D3EL9YX+t2L/WHnv/cKXuZC8LpM
QM8LMwrKkKs0GSuCEjqRFmUjVNREZQjQ2WRBrosY9m2jZcQZCmVg/kWlQRCW
VDYciI2xEUFha5IABbiuoDXoWiqoaEmTY4dBfzLyLlvI+LyYbHOrYU52P/OH
e7+sj2/R8U9y6dPS35PVGxxcVQKkF/EWaFkaaq9pE1Y3ga62qYx84JU6b7aY
1tRUZbHNkriIIao0r+ocAXWZxRWDOuXkPEqkflGe5ykKBLBiMlbADsAQ1DqG
OUXjkpjPFaaaS2rsb9k1zqpxGKOcjs60mJkaRpyIP4wnu+wuLzdw6kkYpl7R
BPAiflBEcZiaalsBikmiFvaIRXFcp3kQb4O4gU4x3sDN1A1YAV7cNBXVmWn8
T5MCZosKjGYweV4hEWKRBTKNoJyQVVT5alyfgOv49/JJV7Qid28vpeZtGW7O
VieXTP7qeOwORjML4IlVE2BeqQi2QeElTZ2mTRgUvAoKEwe5qOs80qGOC6SZ
cbMpm6DJywoK1CRFEgcZSmMQE8jNRDoHkTlPUrj3xjoRmaaYD1+4ojLV8yUB
BraG3ENdMThnLoBOVfL+QrmLxFLeTEKnXZjq18lFU9VeN5uNwX0c1piMHDQV
1CUMc1PD+RcxPkj8Mq2DJDFNkeE9JJGXAUTFmV9VcVkGeVwldZEmeYBAF9Fa
XORbCts8uBpNZWVCAHpPD55T4EgyogLAJhAka4YZp0BYwBTAV3vsRpr0KrOR
l2uz1Y3auZosGFa/T1+AsNCMkShofZj9LCjBt6jg2UXSwvtHTZkmdVjh8Qbs
iTwPII4wjfxy6wd1k4BXFm7zuiy4jmIDCSVZgqgh4UhcUEczaVSUkYQ4ybwZ
6oqNLR2KxEUTg+jQAMnFPXstujpC6T3M+JkzO4X8lXKhYwpCVCR56QcJAWQY
w5hkYdh4RVtmeDQEXawNinyDaMwvY1i7KkSQDBMWx1Wd+HURBCV0pomaujQV
1AspT2XyCBVOuHhIIgvSClqW1jB8ECWD5og4rxpH2UR83jcBROl0h6aYjZab
zDgTtyUz6YNSv76OrGs/QJzE83ADjLLwSz+K82CLemTjxxo6kkfBpq0biADC
q0q/rEK8Q5rZpHUI0YTbqqrjrIInKoE4N0GeqhwxQhbWcVJlEEWOxBLTf0Ae
gKZ4QASKLG9cgc1ItxRDmJ5bTnQ03jcO6MxhOpwtjCCRl01OctybIm0BQchv
IaS/Kpf+F042PiSDkNgvwS0uW+SXMVwI53WA/AV6UtX+tk79pinDoMyhIjV5
GDiXoE1LH88KmxhxWbNp/aaoAsQN8PwK3qdp8niLhyrgbGUFjhnSU9Tc6OGM
hjWaBLg0Ndoy4KKRO+ncsz1QGEqmOxcT80jPCjKd2GATPXZBonXs0a5Wh93o
38FE/xR9qQn2yoJti3QvCSAhpv0tsGXetG2j47BtMhx/A72oq3oTIHCDULab
bbgtNyE+2fg11KSEHoV5UvlQmmTbAD5LtjnEV8VVYBjcVBkD+0RIATwhqIMq
SuM6QMQGQmeNUKLQyFwjEGAQetB0M22TUk6ttahemiHv5FZ0AsUhYbsL2BUG
HbE+bWNurtlvjMeshfc3AVUag3YbAHcJ/SCIvNTfbGNebzehSdrtpgrCTZDX
YdDUYVjW2zAI/NCHerX4KCy3QR6E2025DXN8qQaLOUYIUQYI8cISpq7cAP1E
OlRAoLBh8GEMdZwyNFHNMhDQinKT6hLUNZ6CQ4DwLTJD92xiYkgo7nrTY8Db
KAQx+oXRAZ3Nh12ZfrQfrJjo225/L08pbgO/xF8Tb7Zkz5BetjXPgqBF4AQB
lPAv5FjwBue63WwgmjLYbLfbkjz+ZhNAb8INZAgZhRtcW7zDc2H04KfCLRKi
piyrHLJK4K7qpvbzooYkyxwaxKNE5kA/0zgJOOhRROBAZyeCvThFPsrrfOxS
07HmLE4H72/SqVHmLoLI3LMtmL0w/da+NB1UbRjP+Ymb0D5DLpTRpQWUZVvR
VJFt6+cUCmw3NUvqoA2zCnoRwGRty2oTbnx/u/V9fOAEBIFstyF9ADHhrXsc
/8JtDSULqnrrV7yq67Jumi18UxkAIYDGlU2ZFAZfiPMU8XRWxPgflrBJktgO
nEePZ45QDlEa6DlpYYc0Ef6Z23GnHIcO+m1pLDeKGqaH7gSBOSqZm0DUd5E4
pmE0qe6YfuWDMnqeFEnxI+Qi08DfJM7DwMUwlsN1QDSGDFUTVS2igAT3fFtW
SHLwJZw4ZEPagk+2pChwMqQjw4WPN3hxkOMbhngPpcLToEs+vFEATC2Og02d
1mUalkgk4WBi5EJ+gyi6KlCPrpKkQQoVa51EKgKzENNN0hKahKW0yK8aezNF
MGPGWNIs1YtEEvWNbYKYPFHcT3Q0iZBF5vV4A7uMq6W8irn1z4iTc5wvJS9e
sAl8P9eIy0J/U8kKB0lOv4VnjyGUbUniCK3Fojd4GX0EnekE4ohQzWYAABC0
SURBVNv/SS4BSRDfDmIJSelILnguPBnEtQkrhHAN2cAtwII8AGyQpYipWZXG
KQo6ZV0XOfo1zaYkjxMifsgR/SGYoJYoKnrHqR1zGoNlYxvaUolGGypsY3C6
qS13o2fZAGdAjMm6AThyMf2czFmh1pP0J+iL/YUCSKZFRYVyGZwqqH0b+JzQ
1PDryPxhxZDjBM6Ebdzpb4d323A7/9w+Evrtxn3oQyj0HSCdEpIq29bHdw03
SROGNSDrFqFBDpdVxwDeKuhWFfs17GDC4prHbYaZm00axQCIAEoDO4CBinOs
CIipwB0jMdLazdYm+k2BIhozbsBtThzSHFKD7XJuRCZ6phCTasJ0aPoPkEsa
bHIaBSfLTQt3T5TkACfpgxATtgi48rjGOSJGg5cJSS50/L4/imD+Sf8YKRY9
Ffpk3RF9rw0iafwMyGeD26AKkZ9C7HhaUyGlRWWuws8pUUpjVdlAjI0BcB3k
JmyQqgLBJnuH/rUkC4AKZfAt2AuU5+hey2r8CUALooZMWxpFSQHKGWIHFH8I
RCAPYmxrJ8ZD2nfaNlOjstrRRymHi9Wn9RF+hh3LKrqtG0wRTfNwG7Yt0OWk
DfBfGJmm3ZatX9Q4Jd+5lO3w9vLDmVy2ZOzID/mQA97C4UBQAdlC0iGSUdki
zAthOUtEBFugOQmiA3j+TYCUKM7h0GDkKBBvgaXmFZ5SYmRNGZuK8NNIoG5q
Y3EfLdIpwDk0I9QVSFP4csTApqaxDgbdu1qS71GGYd5NQi1uxUDPBWprurkd
1JogPmvd5ifFySyAPGDHyEuWLZz11legKPn0aI7QDKfol1unLPat/cA5fzr7
a+HABm6sGfPJjm1w/vhHCgdVCVsbFUBl6AlBAFWB08KnPjzNBlKAzQtzMNLg
tuCASnoJEqfQrwFY+7UuAGWjGJfB4NUgGoQJLGzY1NQ8nQGjI9YVVU+JpZtS
6QEDUqOkKjJ4HGb9f4QiAusUxY0pJhaotguFUSr6EXJJqjJzoxRDa3hKYB9J
lzOaIrRq08bhu8rxCdfGxhIQIPxLRWpUhZBY/7UNcLdNWKIoF4QVEtsAXjAP
myAAXQRvS9BBywphnG0ERe0zD1HzASJXFCVkhLoe2hAZGhITENJtg2iOanlk
gTOqXXQzudx8iCzJ+744Iy6H2MkL1NRyWky3QOJT5WJinxx0YCdXZghkYbYI
mInw3gZiKUTjb/4HuZAqQftK0kz4GQroXIBHMTjihRhxRxWQfiEwCLZA7PKQ
AhS6hyBNhGyYAFygJlcn6abRSI1KlIUQysHPULtIlSdxmhJjowDEAOSNqwoh
N8iKQEkV7F4ORoPkKeSkHPNQd62n0zU1/GITB9IdTMDvJqx+Yv3FqnLQtmXg
t0RcZVlNntn3W4SedOOSyco3XyoUCg9cOkrpUYtzpjA8oEfDjZMNogPCBxCs
kV7BNuYUhAMxTUuK0+n8wwYN02VTBUGTNHgAM+lAKQDvLcobwKAF4ooEA1Yo
7KoMOHGZETbFgUTSxmtqsBOJGJ0h36RCt51WbIoLdyPtIDx50Z8obbuOlJ9a
f3E3REQeOqAs39bbcb/6ZNeD1LqeLon/Orm4ZIh+SGjdUkgBAfkVZ0HxKakS
AQoU4sHc4YvwKmFb4S2wCgr0EK2hjldCk+K6iFBURZG1ScsKkoowh7tMRN7U
GaBtyAlz1jCIwLaERF2DqV3B5bg58VAFsqvU5W2Sp/QWG6Zel4tGohKGhMAQ
mojcHaqzhV1D3Ima2JZEgnDo/5KL76Jreg8FRmy9daibjRnwm7QtMFISGaAI
iisCCt/hbgAp4CPA09uY2FTIS5NQIwjYVjlIPDyJMb8LzAQOvhtVvkHUAfcN
TDniynctI1Y0gwzUsAldiOydHQ/OEanL+dufoS8RGS4yFW1Jc3nxKQW4+N8n
75K0FMmGX2zHRvmQK7MwDsoJoXP6hODgEfoEZ1/bEgOMXE5ByTZsULhDGYiM
GsqqNfItsAtQDt1EYFKVIFPlVAKHjpQ1lTGSBBZO5LkC2V0nWTcDqmcLTHel
AI/uBxPfXlinlj/+lDiZFBa4JdmKDWXpdSY1iljW6pO++IjHvlgow7ff+Law
AwGVDv3sNNWn9BNy2RIgB1uGejb4bTWAIYTRDbKYIKhw8k2CIkOaBlWRgO4m
I1S0NzkDMQG1ntIWM5qAg4ILblYCohOrXcduvEisEdzrpwtfTHu6WibYzXgW
nyeXDNCUs6TUlN2Qm4euhBSDhbmXxta0bL9eLv4oF8qWSGUCZ9p8Jzd/Y8No
ehAFoDJv2rLxUWSAl4/xeVOiApoEEUggkE+MglpMHW7E6gBHCmh0iskDmW0j
4Cl1wi+M4HLgjeaXLe3Mu2MS1+zh3R30ZHmxF2bOXABUTO7e+tYgs4N2MlMS
TtKlhYTmNyjsb5wpWXAM9AXf3en0SEinaqMGSkytmB1WYyNf95wNeXf6ktVS
F4b7BG8SRkPqgvPHf9CaZkvgJxw+Cm34cllvmqwKqW4KUI0MWFLlYInkwKBN
UFQmQkdV7sgdSUKzhm0rO9GZs4KG1aF+E0eVok1EVG+huto8QRHDrKiv5o3P
yMoLvD6Q7ugmJH9ivQqc6wbsPgCwhJThovNyUpnhlZseGiPrZzGA0N7ulB/a
19nXEOTiW0yGTpc+IXUM7d1P37eltzUSFkJqKNLwKaNE3g8qAaB/QEPQhc02
1zXR1JD+BxnsFOwSqDU5nLhB4FUQDy0veA4YmhoPbf3fzqJhiZiOfqQtJ2jQ
zGW/0Oxy6Ilc3OH4NbzxrgNVvOdfMrtbJEZLBWSyaSkIID3C8VB5BbdjYGsu
vtWfzVxjOr2hSNYmPmEXSIUu8yDU0rc4pg0v7HPwPzLFloSC50AZK/LhiLvg
1EF5AmcNlgkkgCoh3ieKZwBZkgzzGUHUpX0OKEU3WYbGQOINNoHFUgoX9dLY
QJbjE6TwXQ+26DYEX629UZcDHLuamPpquchxp27/ikufVTQ4JYBiFH7mGWcc
g5NQPkwyADPOAFmrRFrQuWB/7rA732BFYXEvnL+ra9pU3dZcyBhSfcymRHge
oC3ERVRRQE0ZZQQglyAB1KA7IT2Ho0BxDIyZOiHYChAXsMeYmpyZsD2ayLCj
yRQZbqZrGkxkjRGzk2im42Yjc8GjtT3yl/mIdCvVvhwfI5HU+13b/0r7y9tB
p7ASrc3f6OYlCcG10GG2NpUACQYnRYFmGHa532bT64nf+WxK9EJrtAiILq1Z
Cyg3tfYMzoK8eEnFHOhGQ7SmFtzAckNkp7qJIR0gVXEkwJmxK0dzVJa1HTlg
TxOOge5vIp/zcex/P0BT8D5DNvzSerPxjOeZezefboEn+xk9abt7Bl2YfaL3
xXt7EyzMKkDghyWriHTk+xaMcT6Esu1w2wEizquM8ZkVFowX+XBAxgGxMMrS
Jn5tRUEt4PzKhACnUIeskgb3fgTmGISgaQxQYQpqBUOThRZo/MMtbou9+IQt
T4WK2HRJNuFSbKSYZ/IKWfRu0P7VQGCeAijqk1hNuzvci2eXV5dqqR+Z/hZQ
8GKUOhD+o7Vlm8QodTRwpaAjxzHyM0SfSd7EBWjhcQpzU1NqHXaCCSlwhQ8C
1LHFjb9BsbfGt0I7RhMkW2BX+K4V8uzMcsdRSozRw4kyLZh53Xh5u6EBRxHl
dLtPditZUqWczTCh3xZgvrA1R7JbyTC3Qczmli+Hr5NC/q2xgHZok/c/1cXi
IxyhXc1bn85vl3gyQp/WVkgsVyJ0IRnd/RRfbTtHbSOpbVvbAMyCHpaORE4j
IE5ZCboljj7RcBQVjWNEUiEaXebSVIBbE7vTkCX6evmM1w3DTOLIVkYmizSn
+Ef3UtCembqmRcjrSXeLaJaY7gtW3uXi2s+bE3yvXLzq+HQmul5fJJcCz7K0
vVUuP0oufJ/g/06lV7n8DLm4OPnU9l5xlctPkIsYhcNu4TDr9Y28cbnqy8/l
88tVX36MHVMXg/tWufymOb0fFAgeuV5HyZ/M7NTPkstjfwU7YFjMx9fbfvf0
tX/b7Z9/Lb10/8qLn3wtXvzCb335yEE/ek/dN031/Mpdf3z+zlXnV+778ysY
ygt/sjh/hdlYuk7eZQfPA2d7euEHH17+rf/nF7OXf3AnxrtOe/+CyOX+hd9v
/4qH2b9yLvsFEPNeuey9/+mqv+vF3/aDvfKFEKX5v/Zwv/QdX9kT8tIwcfbq
by0/67jkl8aT///YNXG7wPjFvzD7nIakL7uIi3A47fb8uV8SrLnz6fBw36iy
pBRhTgjRyyd/82B3OLJnj3azOwXPBZ8HhCocoT3/YrHggE762Vk3+OXOuXzc
kknZ83b0KXvuNzeYqn6unvyzix3j56d+cJHA6bcNGoRftKMfWUp885N5/hvo
E3/yfnBjr9Ldk54lI7kkT5qz+EwUiGfMoNLQF9xL5vCl6kJaooAI1M9N5QDb
6HTcP5eiuf0u5/JZ/xC+vR3F0/qi9f74uECJbYTk5Q1/+NtXeyeaTx7tkyd/
TPxW6LB9XCjdgAN50E8GROYciXP8LFC22Z+PmyftJ8Atj2QjvjoiIndWq+fu
Wvod4/PToWb8bIrGE/zQ5vhCTNXWT9nfCPpy+Go7JrrxKAbzyZ+TPztHvDk+
HXWcm2d/9eLExbHxnhyhGHnZPnru5541RIr2+i/NKyh5yHanXfv0d0j2u714
7n5Qkr89H9O0p9ORPZsc7g/7/Bl5iuPh7VRSnKy9X3zJC3xCXH/WUx5d6DzQ
h8W7563kT57W/wvk8hH6MG1fVOKi/eGrgKP1ugT0LtumRt6w3ZU09qaoD/RB
erMd5uv1ss5IeSGwbk/lOBPsjsYHuerMV0DVbilPv8/SG5ZVytnid7Fasv9T
NNaNSDUn44++ZbILXv44/PuPeZdLCE0x5/Blv6eSDaNbRwTVvC8XxdfjfVEy
fN6EI8rTmxkWW9o9lXkP2Bdpn9AkR/auqvRLzdfrSUQBB/8mZ3anMBeoWDIg
n007dL2de2riaNNUN9nFbns8Mu+1qtp6vXnMP+2wvfR42p8x5MBuPg3O50Pt
XAzIXCE/nnaJOL3tKyAIACSTvQPg+PGwr7x6dzo6LK7V7fm8a7zj23mb7Q5o
JF0t2hOX9SNvsjozVMAAcJoDbdZE7cQ77iPRbabNoRvB0csOovbtNOuqlYRL
ktDaDa1L2AE8riBfrzp7xyNP98yWUHK20Pm6Xnf6F/WG/jUIIiac0dYYD6iQ
lj70J3bdsDhjwuz3GfUfZkdqC6XH6LKd1SSkupUH26R4rkkDI3rohOGj3upn
nkWND71cIJNjbg2bJ9u2+8QVCqj2uI+oPHusJaqYfQXAyQWfQJXwshr6EnP5
ZnUMa4V28XrAL/iX3o4Jfahgdk7oQ/bPETtEVqWyvSI7hnAgOXoKWlW/Ob/v
dXYs2hmJksAhM+3RCvNNFGDBYkhi26ze5dkL/lq0nd8/nwpRAStvebC3ft9O
3NzvAnY8nDDbcLdvitPhuJcuRpOj34cmVYfz2WctJHsS3mFfhof9OXqNwPZP
27HJGio9kDzItHVfmSCQXT/9jKAj57xeNgVqBIE6Kybz4oXWhqab/cGCQK54
yg/KMzsIhiAxBod+2p3Oi4ZIoK36tNvyVW7/U+QsLutkfIWLfwiSqSajVtkH
CrFK6H8mjF9vP75FEFiv/7EcI9WqET8xeL7/aSti/D/IQ17MVPf6IdF/wKn8
B0DRcoonH1IHAAAAAElFTkSuQmCC
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 5</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see that we will need to set a higher threshold (which makes sense, as you’d expect more UMI’s per cell rather than unique genes!). Again, perhaps being a bit aggressive in our threshold, we might choose <code style="color: inherit">6.3</code>, for instance (which amounts to around 500 counts/cell).
<ul>
<li>In an ideal world, you’ll see a clear population of real cells separated from a clear population of debris. Many samples, like this one, are under-sequenced, and such separation would likely be seen after deeper sequencing!</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-mito"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: mito</div>
<p>What threshold should you set for <code style="color: inherit">pct_counts_mito</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-5"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-5" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">pct_counts_mito</code> would do here, however the scatterplots are likely the easiest to interpret. We’ll use the same as last time.
<figure id="figure-6" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAYFBMVEX////1
9fV9fX3v7+/5+fn8/PyCgoL+/v6AgIAzMzPp6enh4eGKioqFhYXa2tqzs7O8
vLzFxcXU1NSrq6vMzMyRkZGjo6OXl5ednZ11dXU+Pj4fHx9hYWEqKipPT08I
CAhxVciEAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAgAElEQVR42u1dB2LjOAwU
RdK0SfUut/z/lzcgVW05cUkuZaW7TXHsFEJogwHgeQ9cymPeen3xJTz56Evk
emr/x/XwMSuxHtrXq0v/5rGXrNdXX4/e/Wp1Lj/QxazK8n8JRSK+eiQWM+uZ
/Q8Xe1QD+O60W68vvw5sOSKT7lHV/Ztcuw9VSqn1fn/12t20Vzhf+3bmV9Qq
l++Vi9UZqSAP5T2mL2qVy1fKhVRFdhaNSXqr3Ce79dC+Uy7SWrHefgk5IAOr
XL5XLl24RqIRzqrd6fetLVtP9uvkojyp94f91tsedru8F9XHclFSrtDmF8pF
esKkntknQTDBN3cfen0pxQoLfJFcpHMxEI23T/xgjJXZ/o7yTKbXk/2qONn5
fe7FOxMeTkdDn4rmdHzzPq4acJKp10d06/WJ+uIiMOVl58rT3GuPXWD2kb7I
IRldncyX5ZW4888b+7E59Mqwex99tn6JXim0Xr3MF+Ewnji2+AQHHPT6Ij/U
l+7/VS5fpS/QluwNIXJ13O3OmT3zj/P9EVJb7djX+H3ZhWBqLAXQiZ+89zkB
BHeq/gmraL5AX1SHiPVicRjmh7hlJ9JRcdbrs+3Y57xivVa5rHJZr1Uuq1xW
uaxyWa9VLqtc1muVyyqXVS6rXNZrlcsql/Va5bLKZb1Wuaxy+TK5qK7ctlb+
f4pcXPeFoqL/yl7+OXIBO1ZdEjPW60f5l1UqP04uUrKxb3O9foh/UeD5R2x1
Lz/Lv7gGDCNWWtnPs2MuHFvl8nIfn21Omj92WseRfHs/sloIotZ8/5v5/IMs
5EyBVrl8t1xGYYjJ5MTdGk59r9+HjxZ6f9q3nj4fjkXfN7Ff04/vlQvBizpi
5py2pWw3q3/5KX6/m3OxS3eZZ3Zdpv5xP/J6fXX/i30T7WxvpWtDbk7nt/XQ
vj2vhIKYfW5l8tanLY/aMTQzv6uR3tLQplUuH+gLP/pIJSPPnOwDD89TEsJw
b2FUo3RdZaqbDpSskniof59RPzJrSy/wPfmc32fvDdC0HX+klaskHuxHPu0O
abQ/nBEn0xE/Lhe5DIcx8fzA83X+GO9zS5wee0Zf5A2YkhntmpbdfKDVvzyg
L3aEL+vvZ+XdOd/y/nDAVWJWdXlALqKrKo7YmHohr/xgxAJfZfPAXIW5AVJP
5vvqY68jmnUs1tfWxdQHVbEhPp4JSPB15P9XyuWugTD2SZNnklYysfi0f5kI
8GlymRdpFrVFkap0leR5BrOuAPpKfWH3naJQU9xFSakWBSL/aXDms+SC+I1H
5rZkbIbSz4lVox4oN9ZcLc7X/IcnZf1f+gKZqSE00PIajFkInv/l1Oaz5NKN
5r15kHycjS1YwT8M4yTL2L9cF919ZiXtnQBKXarCBO9Zvsw/TYfaLe6mFN/X
l6RWavniKbO1X+wnykV6+rjfHfUql5/mX85hEYXn75KLXHtkbujLbiAhf5tc
1CqXa/+yR6292H2fHROrVBZPOd0fDrvi++IxzVa5LJ2ykVww821yYZFe5bJw
ymr3AUlsd0kDmEH8qiM2u4Wl6gNXIvSz5YJ/TS5RfsjzvH7EvyhXUunPU8kB
jRQf+HfQnxcBznWR/NUpx8e34/nox3fri5CTYfGOc+YEAkFF8gNcRrJlga0a
c60YyQfn8kHnHx8hYpaJZ8gYbK0qX59yKdoj/Xe/f7E7eHjfwMRTbU2YxTA/
OGC1EBELxaKIrYbsyo55edPUeXXvKxwDzNi4WjGdelz1bJePnP7NiGzVlyWr
9JGB3y29QPTMFtXbJ/Whm1isILsXryHZQl55Pu1Oj+Aw/eEaJmDE+hhAKrm6
78+UyyHJjM7ufoXqCYC0T6yrJIteXcSHVqzzRPI2e/yhcTLy7+JjRzPbj3x1
tKdbtUh2qSBLCiNcoHAxokSKWN6EkYW8a5LG39YX4TVtg8zSunF0vhxOEuuR
T6dETF4hb93UaqRfWHMm5R33tRvo4y1zyO/fU6b+ev1lfz4e+zg5TvCENnhf
XyBB07NcGZcAuNwh8YX2F6nmiuHG+HmO8reoLfeFAdLBDn/XjvEzn0ZZGmII
gulCvv3Fnaks2NhhL+AjCcPsUQqdMSmzjF+d6zTL7+TisVtnKu+nBUj1p/Ul
kv2ad1wZepHbw/5snB+vrvuR5cgKkD1jr0O/+g/NlJgkpY64nC/9e+f8Aerw
e/4Qwf4UfrMQj6H8cnJOBB6X9IULXh7lUDe7CAXUeLP35NWuQ2OwTBcw2Oyg
pXNNuhAUlC2hNSa6I9GUxrC/LRcTGVzWOxD06J6QnQafcM1s4plrOVYjd1wO
hhCIv5kdmJwF0PD4qlMudcOOybvyf8bUX2oA2C34WWfDTvYwo732Mk+UR9Y3
qO6uqyjoX5FdxiLVJIaSnaBvnDcpmkzYrH1fdaEdS8VV1P0vUcl3SxGQtS0U
d+nj4W0fHncnWo8sFvsrbblEXp6c05dIv+eolaLe/usePmkr/ObC6bB/iyaz
WOWyucduDJPkwMhfyPflGM9aZRvPEsdeiPcCKGFnYYipHig5NI/PGmWolvMv
dZQv8S2dJE4XiccNPNkN9XM+RRRmNlCOyAK3Yq0ujZRKeN5FP4wtE5CEJuoh
/nG5DPe3OOvLGQhqqf7SJ4VWKLzv/J/4/nci4B5LuzRXY+ig7ufn/gP1yryB
dY+u8oib9ReXHDI2VaxRqpH+sBYzWrtEj6jnAsAm/107dmbp/ni+jU/tPmgp
p3FyONqkozoxrnXK3svT5fT4eReTkX2bm1H5T/H9rk95J5vtHAST771CXuDw
lGYaM77EoL9Iv1dHvlAKJVRfYJZrX9IUhykxnu9hfnLEe+kwAgiGpRVqKf9Q
g/WSF+2Wc0XyvFUuo4UPai8LHpWLZn1aOOpKLwbBR7yti6P5DKdZlIBc65Wz
q6ETDp7oR+7azSIxCW5t3ZPeutu/b5nk6XD2fLk/TfV8wVUunZ/eDc0Wz/GT
2VWHchR1ysOMzVqljda6Q8/MjT7ada7C9KrAt9wf9/sXeeOyH5PlYYEFdy7E
ymWkWlq/f5ObITO27Hf+UbmY6hinSSqfnbjsFENoiMPhaRmfuPEeoxy6wm4G
XMJ4nlzl4s2auPmrfa/STUgSo1kbwS4AN05+uvsCNws/L428sXKwygXnmBze
Dm9vT0z0H4CyXglEkl3h9J0vp1GjhQMnRZrxJS8l/+kuy4V6ZfbgK8iNJ/TP
eYQJCDnD+aWzS3LgXthhigMGozrgRU0KMTqlPhr1L/r/67rYmT3Igx25ZvYN
u6YNyVGPXJzVkcrhhmb4pxyn9diYAGGbnVkiVn1Bvt/WeR4/jEDLTjImHk/R
FPnY9y2GZ9rTFxHzlvJ9OXQnuY/0vzkA41ouLdHHjg8zze3trS6H7zA5HSza
OXzHFx+9Pese9fovdwZPDZN8Vzv28RHsFvj3XfQLYjMS+dRMv4dG8cYCyoIN
2IzqrZprAyiivkI58S9ePwxT/JNI2ZVc9OP6wkYWmRo8vXEf8bjQNr0XUcGu
2sQopUncYgVn5mQR8fndIdUaj9krzuPyfrnQTPKUyzGpJGiFvJPWjjfGhqIa
m/Aq+7IOIE02TOeD+GRX458VxsRqxzqDwu/HYWZgV3cVg9+fxmYUWEWuRKZ4
1CEAuQPJtG0tn4QMrmppY2ulVjvWHXX2/HwYe7RLmaKXcGSarCPh6ek0vg4e
sKpja9Jq7Re7chaU7p/y5zf5iEi5BvGBDasuMxqTORGsc5Me8+LiFXxM6aKj
K7k4wtoyM0Pzmb7AzOZcnLW7cvGU43Ybv4Jb2hR9LE7yiSRmjshNuB7KNXry
BalWuVxex33TnNvX8OSObwYujBUEpymiU0STZaoHXQayvuuZVeNkk1UuswtE
cY+/Nn/MVKIr7GuBnFGkQhSRG35h7OxkmQ2sVyPkBMZh3jqv70a/2A5yYS/O
6+ODHWOFzVyUHRsDYxWbC96Z6A0YkwmfVAJWuVxc1eF4POQv+H1ppo1g/AbP
VnWWDPxZpzAx4x2mJlZ9WTplE+fDTX08QHNoSbIeqsG7SwxNXS2L83ryCzIT
yljkyEdyzfrIXLN0EggUE4EZfc94H3XZg6Rubpib3hTq98olJ8PT60uaUj9y
rRt/nBu3SMCTinhjko+C0qme6QtLi4kUk2mQzMZOPdk1ql3p1+0S6X07gX5Z
FedaLic5bQbnB1vBpPbX7pT3S637wLaqgqCyASVjaapHapltPSvGzmE+O0zj
TdhNcja939ygNi+t8vHe2+vHrlc1/TJ+Mv2Jg1wIkTmofj2yyE/Ht+v2evuR
u+fpra1CSkvMZ+MYv+mmStblmcLCmLLrXhK9IYvm/NrFkE+zh6oTMhO/vI58
3KSZP+DJBk+Arsi3AQy47BdjM4I3baHg3ZkxlcVqmT/BHQajk0Ep5EQyE7mw
G+VKzS4WXUFV5O2BNL9fLp729/t2AGNol/hb1AGZcvAvY3lEg2s2hAkWl6Sm
byrzR6lxEYDolWYyTHw29EXbJEZ7LwxRZJGRH9BAf7cd66sjrHVb3tE2toXf
HwGA/cXJETeCUZGyR4Q5vdNplonOa4gLSgt9lER2s+jYnClYxs3dW3y9q3VK
nL+7YOn3+/3+OHb0Bv3Iu9rsdzs93yuqLjnjkwkW2ayVWJIl47ZWPNEGNWUx
2epXAc9EVs2kM0loeR9P+a4A+FfHyeN8HnlnZWAsZzFsCM14wi828ABTng3V
SYcnxPOuMNXTAFx7jLBP7euW6hGw2f684WcuDlwQ0Q/2ObtboOPN1qTdVZu4
GCsEWsoUYIuz+IiL4z7ymp3MCPNfRcEmZbPBDGOjuho2LBpzx33vpjQy+UsH
ad7UF3G+ofW75dVhBBIzm6TAUZD1KbJEM20yEk9+NbstM+BhCDNjVhgzzse2
BQCp8l6iYrqhxvDkvr8O8wPeH4Emf1FfkgXJuvf3vCLr5r8QEZyhKMaNoNQQ
qoO737CikPOJijJyfkMwbiRLu0qyNXe9fSNuGYqeCc+VbQm0srFjNdRktNkH
jR4kESZulnKU+skVuOu7/0hnsr/9p+8W7Z52YCMv7Age9+dm8OV2PiKLeEdS
8jIhCOS3cVuBdhgVuSEZNA9QaR5JkblsUnppzygfB8yOPJl7m3DeqbD9aGL6
1fzk7Vu7PbanR/gwJJpUT/kVFDVDVzyglhUJwfA+VYwsoNlpicNOsrxIOn2i
KlkWLRHXrbMQ91seMZ33+wvnLF3KpSh3ZVPH+pG51vwqZC1Sm5GzgjbAUpAl
00kSg2E9U2qTqQo+BLFscvzCm1GaaZ/sIyGZWoQ35+7lF/kXPd3cKu+xfLFF
9NNovFcJiCTXXMRo37dEZD2eE0/SVAzjmfqgoA+hzRSFEc6fSDkZwCDv6ruQ
Qw+UWt7A/MOHYS7UkalhbP/A/GRnvIpJzytrSBQhHoTJqnsy2YjCsI5GFs3u
WZPgS1kmi/nxMTc/U02p/8r763swlvHkh+YqSBruxpAAJt2dXqS2HYxBMxBD
Rzb17LsmKssgL4xt3ytonn+X0SfJ6HX6K+e8su0vlg8w2/2zYLSY/Nu8C2SF
8d17EkVXAxMGpDHEUjl3kTMH2C8bnXcYGVeT7bp4fuRFSaokjB11yNqAI+2+
OPb/G6y2UPMJ2fKWE6dpjkYK9YfnXQAN2yWPxGNZRhgk7n1RcRlrSc0WLNO2
H2bkulKP0pjF2KoZ9pixkesnhwbxThLpdGS2wCAfId+b6cvU3+Zbtuf36GML
+8WE5Ycz9y9Ki6KmFIhmZXkVhcVMdvXhftLlBAZDUO34F6rjEhYDzCA63RL9
iKWhM9d42TJ6JOUf7uPbxLG/f2xSKSFjuY2tE1MkEdoAdczcQRqe0UwxKwbX
n8ym5ZeBPg6TZqWWTUAUig9koWmkH+TfjcDiBGgWvx7If4rXpx+bd2ENC5kk
kbE8k5WhuDjiAfBFrB6hnDKyjJciYd5VbjhW6hs2xU1zWx8YgjiMLreaw1J+
O49Uf1hfjhmGVJwfrKTZeSNphnhWONfUGJP37Jbu5oZCFZHBl7kjxcp0ANbc
UVdR13RhGCAZPcFKrCHUnvcuNCb/dJzMdm8HtFqc9vIxBBp+3lBJufAs2CuI
h0aJSmP6Gx7YVwUcE4+bqc7AryQ2M/W6LmiDNk0vy60dvCoOq8vtC1xeDpaT
f1NfMswbj4wu7vT7LveLeV8OLmLeDwgXSZrHbhgpd8WAbMD1scRH9fiCzKLJ
eBmyU1nn2VkhL4oJySCpDmMu5oOW5R/p1Ng9PJFtt3QHs7HbpYjHhgvbvh9Q
j3JiScuJTWeo6UJrMf9hpncsEeowPIlSoUa43qM9GZPxV/PC1tX83j8wD/6W
XG6X1RdewSMXfKEMVTTUYUxk1gzd4cJy+gn8xwaS2GYkhcnSNKvmU01pSgx3
pQFhaUuTPU3ZoDMusxFDVyzSTkHFZv4HO5l2D6/rusKTdc/pswXkOCKbBblE
xGGCLTNJLnKCBNKsAPGZhCWiCfpFLBhWdHd56gYEi9mQK23cSplILpa1xAz9
/iMtTbvXXwFnMLUuGTw9QuNNYheVGMTJccm5zqvCIiyZy1V6QlllBy+qyPp7
+20QUk/5SoDZtEkd5qwzraYkdM8N851NkFF/o39m9/B2u93i3FY1MmB0biT1
vlqaDPxKpaskilIXADg10IkciMyScTACPMxfcDxzWMEJocabLH+RaTZwK8UM
QZ5nlH8BwNx93I53zyuoBRwYZQxTk0WoQNqw175Fx34a6ULXsZ0ZW5sKjbGI
btWwi0GnBoJA1pJ6VLhBhsOb5aiqW1jGbQymFsF9O76Zr3asBzsjtHvhtBKw
lHIEy8CJa9bN7BM6j7UA5p/BEYC3BGYGT2x5mehjyCm50bZOxm1/BwcM2vkf
l9aw68FbcsrUHMXnUhfB+RAmRtnYwUE1sou1Dn9eLl5a637pq1dUxB0XyAyJ
oJyYPHGoc1ZQxpnAoiVaK+N4mYQ7A/RnXWDH85j6mqQVjTtAO07WVaNt8G3E
uNsPtP7pwPO50+8GNAj5C2Pnu+Wi3sNh0HKE0wLQjsEVeeFlFUpfuJVjoonl
gMUyWKcMIQAaZCCcnNdc6iQubSxGzf6W3czdOF+3E0vHl1RYIGOxdpNm1MhX
4mLGmBn2All+zjC4QYylZCl/Rdr5CXIhV0K5IxGOU1idqGRenVK10dAMH1gl
KIhJogJsl4TH0uuTHZRtMnLmZly9Q2KidAeJ6HCQ6XxQWT9f1u2ClxM9mdYz
Mz3n18rOkv0pfRnKUWyhLqYs+E4jxfPaEVTTJhHQgMISw5G78wg3ehRVAPwN
7bpQiYV7YtunnGqAnZM2jAJEjeUxGM7R8HSyzqRjZUg12zY3bgCEpVOjLnUG
71fskb9r6m4/pUrd2P9i4yqB5LsKKLnkmavFeMQij0hrUi7SFGykjCBndFRQ
BumYSHHuZikhA41cjkl1zMR4006lDm7Juj40O8OpYG677NKuWEEFuAl+KXWm
huCNUZaq/oS+CGy0pM2iyvVfXDY1IueA1+dtTohY4qXUzszdP0BdpfUUOcIw
rWtd1gWYsbXNF3MTc64EoS7gxopIi6ThgNdw0EOLrANEeyDUpTiiM2dSmGyC
HPFxShDj873NoutPs2U4zuVf8fsn7WYjqIVXENWVp1v4A0pbihLSSZDkBzF8
BKooddaUtPYzi3QcR3imBlOm5g7kRB5PGTtCBa8O8jwBXBklRA6w511VPe8p
dydOhvJiOrwcfD4fV50s58Rkwi7R0l9uxw4jMsL21/qCtNDYJLIKOJOKe3mJ
yjHKlmXtNRtEY6mEGymLFF8tWWldOp2iTLsMpILBK7MC7H9tJzEWnclR/czy
xFolG02nF2HacOxcLG5/V9OVT+mUR65+tVzo5A/7fW2FUl32I9s/t8L5ezIH
it/QmQclwmUuorISJq+jSmeo6GRFmuasILw+q20ag6QvQUKaoAlG5mgE4HiW
MHGsEcKlHTM2gaSI5cxd3yYdpunKAkl/7w88jetYoaNaushAXG+eV79aLjTC
JTtbap4QbKn0nwZNATylAkE/RVKJsqSnsd6nCGLYs7ioS52nhGAmuS2XkamK
kzQ2tWmApWX4QoOgOYkRMqA8UCVEeUEeww1kyQlhBv28Q8KQKlGjBixePi4x
leOQjJm01Mic5e4v+S0Vgd1dqQv+sGMjev9y/afpLA8SskUlil/lBtTLIBNV
A2p+KvMaBJltngFU2TYV8GSSSFFn8PqRCeLCq2KQaGNqvojtrA3cAIjRUMXR
eSFNEaVA/6MYGLSwo2UR0RGxg8rUo30CJGaxNqUyPdsU2MXOlg9qJ6KobogJ
+9kKs7snn4TN1+e8W+qyv9GFXZXAHQtEX7oKKlZUBYvyBKqAsv8mSEpAkVFu
kqRAK1KeBjFrKnjxPG3ShBYp2hgYj1BN3yu0dVvgCOI7VrFOaeWFtpUZiR2x
GcnH66suEZsRybHBNxVXa+iznq7R73Xk9D3EL9YX+t2L/WHnv/cKXuZC8LpM
QM8LMwrKkKs0GSuCEjqRFmUjVNREZQjQ2WRBrosY9m2jZcQZCmVg/kWlQRCW
VDYciI2xEUFha5IABbiuoDXoWiqoaEmTY4dBfzLyLlvI+LyYbHOrYU52P/OH
e7+sj2/R8U9y6dPS35PVGxxcVQKkF/EWaFkaaq9pE1Y3ga62qYx84JU6b7aY
1tRUZbHNkriIIao0r+ocAXWZxRWDOuXkPEqkflGe5ykKBLBiMlbADsAQ1DqG
OUXjkpjPFaaaS2rsb9k1zqpxGKOcjs60mJkaRpyIP4wnu+wuLzdw6kkYpl7R
BPAiflBEcZiaalsBikmiFvaIRXFcp3kQb4O4gU4x3sDN1A1YAV7cNBXVmWn8
T5MCZosKjGYweV4hEWKRBTKNoJyQVVT5alyfgOv49/JJV7Qid28vpeZtGW7O
VieXTP7qeOwORjML4IlVE2BeqQi2QeElTZ2mTRgUvAoKEwe5qOs80qGOC6SZ
cbMpm6DJywoK1CRFEgcZSmMQE8jNRDoHkTlPUrj3xjoRmaaYD1+4ojLV8yUB
BraG3ENdMThnLoBOVfL+QrmLxFLeTEKnXZjq18lFU9VeN5uNwX0c1piMHDQV
1CUMc1PD+RcxPkj8Mq2DJDFNkeE9JJGXAUTFmV9VcVkGeVwldZEmeYBAF9Fa
XORbCts8uBpNZWVCAHpPD55T4EgyogLAJhAka4YZp0BYwBTAV3vsRpr0KrOR
l2uz1Y3auZosGFa/T1+AsNCMkShofZj9LCjBt6jg2UXSwvtHTZkmdVjh8Qbs
iTwPII4wjfxy6wd1k4BXFm7zuiy4jmIDCSVZgqgh4UhcUEczaVSUkYQ4ybwZ
6oqNLR2KxEUTg+jQAMnFPXstujpC6T3M+JkzO4X8lXKhYwpCVCR56QcJAWQY
w5hkYdh4RVtmeDQEXawNinyDaMwvY1i7KkSQDBMWx1Wd+HURBCV0pomaujQV
1AspT2XyCBVOuHhIIgvSClqW1jB8ECWD5og4rxpH2UR83jcBROl0h6aYjZab
zDgTtyUz6YNSv76OrGs/QJzE83ADjLLwSz+K82CLemTjxxo6kkfBpq0biADC
q0q/rEK8Q5rZpHUI0YTbqqrjrIInKoE4N0GeqhwxQhbWcVJlEEWOxBLTf0Ae
gKZ4QASKLG9cgc1ItxRDmJ5bTnQ03jcO6MxhOpwtjCCRl01OctybIm0BQchv
IaS/Kpf+F042PiSDkNgvwS0uW+SXMVwI53WA/AV6UtX+tk79pinDoMyhIjV5
GDiXoE1LH88KmxhxWbNp/aaoAsQN8PwK3qdp8niLhyrgbGUFjhnSU9Tc6OGM
hjWaBLg0Ndoy4KKRO+ncsz1QGEqmOxcT80jPCjKd2GATPXZBonXs0a5Wh93o
38FE/xR9qQn2yoJti3QvCSAhpv0tsGXetG2j47BtMhx/A72oq3oTIHCDULab
bbgtNyE+2fg11KSEHoV5UvlQmmTbAD5LtjnEV8VVYBjcVBkD+0RIATwhqIMq
SuM6QMQGQmeNUKLQyFwjEGAQetB0M22TUk6ttahemiHv5FZ0AsUhYbsL2BUG
HbE+bWNurtlvjMeshfc3AVUag3YbAHcJ/SCIvNTfbGNebzehSdrtpgrCTZDX
YdDUYVjW2zAI/NCHerX4KCy3QR6E2025DXN8qQaLOUYIUQYI8cISpq7cAP1E
OlRAoLBh8GEMdZwyNFHNMhDQinKT6hLUNZ6CQ4DwLTJD92xiYkgo7nrTY8Db
KAQx+oXRAZ3Nh12ZfrQfrJjo225/L08pbgO/xF8Tb7Zkz5BetjXPgqBF4AQB
lPAv5FjwBue63WwgmjLYbLfbkjz+ZhNAb8INZAgZhRtcW7zDc2H04KfCLRKi
piyrHLJK4K7qpvbzooYkyxwaxKNE5kA/0zgJOOhRROBAZyeCvThFPsrrfOxS
07HmLE4H72/SqVHmLoLI3LMtmL0w/da+NB1UbRjP+Ymb0D5DLpTRpQWUZVvR
VJFt6+cUCmw3NUvqoA2zCnoRwGRty2oTbnx/u/V9fOAEBIFstyF9ADHhrXsc
/8JtDSULqnrrV7yq67Jumi18UxkAIYDGlU2ZFAZfiPMU8XRWxPgflrBJktgO
nEePZ45QDlEa6DlpYYc0Ef6Z23GnHIcO+m1pLDeKGqaH7gSBOSqZm0DUd5E4
pmE0qe6YfuWDMnqeFEnxI+Qi08DfJM7DwMUwlsN1QDSGDFUTVS2igAT3fFtW
SHLwJZw4ZEPagk+2pChwMqQjw4WPN3hxkOMbhngPpcLToEs+vFEATC2Og02d
1mUalkgk4WBi5EJ+gyi6KlCPrpKkQQoVa51EKgKzENNN0hKahKW0yK8aezNF
MGPGWNIs1YtEEvWNbYKYPFHcT3Q0iZBF5vV4A7uMq6W8irn1z4iTc5wvJS9e
sAl8P9eIy0J/U8kKB0lOv4VnjyGUbUniCK3Fojd4GX0EnekE4ohQzWYAABC0
SURBVNv/SS4BSRDfDmIJSelILnguPBnEtQkrhHAN2cAtwII8AGyQpYipWZXG
KQo6ZV0XOfo1zaYkjxMifsgR/SGYoJYoKnrHqR1zGoNlYxvaUolGGypsY3C6
qS13o2fZAGdAjMm6AThyMf2czFmh1pP0J+iL/YUCSKZFRYVyGZwqqH0b+JzQ
1PDryPxhxZDjBM6Ebdzpb4d323A7/9w+Evrtxn3oQyj0HSCdEpIq29bHdw03
SROGNSDrFqFBDpdVxwDeKuhWFfs17GDC4prHbYaZm00axQCIAEoDO4CBinOs
CIipwB0jMdLazdYm+k2BIhozbsBtThzSHFKD7XJuRCZ6phCTasJ0aPoPkEsa
bHIaBSfLTQt3T5TkACfpgxATtgi48rjGOSJGg5cJSS50/L4/imD+Sf8YKRY9
Ffpk3RF9rw0iafwMyGeD26AKkZ9C7HhaUyGlRWWuws8pUUpjVdlAjI0BcB3k
JmyQqgLBJnuH/rUkC4AKZfAt2AuU5+hey2r8CUALooZMWxpFSQHKGWIHFH8I
RCAPYmxrJ8ZD2nfaNlOjstrRRymHi9Wn9RF+hh3LKrqtG0wRTfNwG7Yt0OWk
DfBfGJmm3ZatX9Q4Jd+5lO3w9vLDmVy2ZOzID/mQA97C4UBQAdlC0iGSUdki
zAthOUtEBFugOQmiA3j+TYCUKM7h0GDkKBBvgaXmFZ5SYmRNGZuK8NNIoG5q
Y3EfLdIpwDk0I9QVSFP4csTApqaxDgbdu1qS71GGYd5NQi1uxUDPBWprurkd
1JogPmvd5ifFySyAPGDHyEuWLZz11legKPn0aI7QDKfol1unLPat/cA5fzr7
a+HABm6sGfPJjm1w/vhHCgdVCVsbFUBl6AlBAFWB08KnPjzNBlKAzQtzMNLg
tuCASnoJEqfQrwFY+7UuAGWjGJfB4NUgGoQJLGzY1NQ8nQGjI9YVVU+JpZtS
6QEDUqOkKjJ4HGb9f4QiAusUxY0pJhaotguFUSr6EXJJqjJzoxRDa3hKYB9J
lzOaIrRq08bhu8rxCdfGxhIQIPxLRWpUhZBY/7UNcLdNWKIoF4QVEtsAXjAP
myAAXQRvS9BBywphnG0ERe0zD1HzASJXFCVkhLoe2hAZGhITENJtg2iOanlk
gTOqXXQzudx8iCzJ+744Iy6H2MkL1NRyWky3QOJT5WJinxx0YCdXZghkYbYI
mInw3gZiKUTjb/4HuZAqQftK0kz4GQroXIBHMTjihRhxRxWQfiEwCLZA7PKQ
AhS6hyBNhGyYAFygJlcn6abRSI1KlIUQysHPULtIlSdxmhJjowDEAOSNqwoh
N8iKQEkV7F4ORoPkKeSkHPNQd62n0zU1/GITB9IdTMDvJqx+Yv3FqnLQtmXg
t0RcZVlNntn3W4SedOOSyco3XyoUCg9cOkrpUYtzpjA8oEfDjZMNogPCBxCs
kV7BNuYUhAMxTUuK0+n8wwYN02VTBUGTNHgAM+lAKQDvLcobwKAF4ooEA1Yo
7KoMOHGZETbFgUTSxmtqsBOJGJ0h36RCt51WbIoLdyPtIDx50Z8obbuOlJ9a
f3E3REQeOqAs39bbcb/6ZNeD1LqeLon/Orm4ZIh+SGjdUkgBAfkVZ0HxKakS
AQoU4sHc4YvwKmFb4S2wCgr0EK2hjldCk+K6iFBURZG1ScsKkoowh7tMRN7U
GaBtyAlz1jCIwLaERF2DqV3B5bg58VAFsqvU5W2Sp/QWG6Zel4tGohKGhMAQ
mojcHaqzhV1D3Ima2JZEgnDo/5KL76Jreg8FRmy9daibjRnwm7QtMFISGaAI
iisCCt/hbgAp4CPA09uY2FTIS5NQIwjYVjlIPDyJMb8LzAQOvhtVvkHUAfcN
TDniynctI1Y0gwzUsAldiOydHQ/OEanL+dufoS8RGS4yFW1Jc3nxKQW4+N8n
75K0FMmGX2zHRvmQK7MwDsoJoXP6hODgEfoEZ1/bEgOMXE5ByTZsULhDGYiM
GsqqNfItsAtQDt1EYFKVIFPlVAKHjpQ1lTGSBBZO5LkC2V0nWTcDqmcLTHel
AI/uBxPfXlinlj/+lDiZFBa4JdmKDWXpdSY1iljW6pO++IjHvlgow7ff+Law
AwGVDv3sNNWn9BNy2RIgB1uGejb4bTWAIYTRDbKYIKhw8k2CIkOaBlWRgO4m
I1S0NzkDMQG1ntIWM5qAg4ILblYCohOrXcduvEisEdzrpwtfTHu6WibYzXgW
nyeXDNCUs6TUlN2Qm4euhBSDhbmXxta0bL9eLv4oF8qWSGUCZ9p8Jzd/Y8No
ehAFoDJv2rLxUWSAl4/xeVOiApoEEUggkE+MglpMHW7E6gBHCmh0iskDmW0j
4Cl1wi+M4HLgjeaXLe3Mu2MS1+zh3R30ZHmxF2bOXABUTO7e+tYgs4N2MlMS
TtKlhYTmNyjsb5wpWXAM9AXf3en0SEinaqMGSkytmB1WYyNf95wNeXf6ktVS
F4b7BG8SRkPqgvPHf9CaZkvgJxw+Cm34cllvmqwKqW4KUI0MWFLlYInkwKBN
UFQmQkdV7sgdSUKzhm0rO9GZs4KG1aF+E0eVok1EVG+huto8QRHDrKiv5o3P
yMoLvD6Q7ugmJH9ivQqc6wbsPgCwhJThovNyUpnhlZseGiPrZzGA0N7ulB/a
19nXEOTiW0yGTpc+IXUM7d1P37eltzUSFkJqKNLwKaNE3g8qAaB/QEPQhc02
1zXR1JD+BxnsFOwSqDU5nLhB4FUQDy0veA4YmhoPbf3fzqJhiZiOfqQtJ2jQ
zGW/0Oxy6Ilc3OH4NbzxrgNVvOdfMrtbJEZLBWSyaSkIID3C8VB5BbdjYGsu
vtWfzVxjOr2hSNYmPmEXSIUu8yDU0rc4pg0v7HPwPzLFloSC50AZK/LhiLvg
1EF5AmcNlgkkgCoh3ieKZwBZkgzzGUHUpX0OKEU3WYbGQOINNoHFUgoX9dLY
QJbjE6TwXQ+26DYEX629UZcDHLuamPpquchxp27/ikufVTQ4JYBiFH7mGWcc
g5NQPkwyADPOAFmrRFrQuWB/7rA732BFYXEvnL+ra9pU3dZcyBhSfcymRHge
oC3ERVRRQE0ZZQQglyAB1KA7IT2Ho0BxDIyZOiHYChAXsMeYmpyZsD2ayLCj
yRQZbqZrGkxkjRGzk2im42Yjc8GjtT3yl/mIdCvVvhwfI5HU+13b/0r7y9tB
p7ASrc3f6OYlCcG10GG2NpUACQYnRYFmGHa532bT64nf+WxK9EJrtAiILq1Z
Cyg3tfYMzoK8eEnFHOhGQ7SmFtzAckNkp7qJIR0gVXEkwJmxK0dzVJa1HTlg
TxOOge5vIp/zcex/P0BT8D5DNvzSerPxjOeZezefboEn+xk9abt7Bl2YfaL3
xXt7EyzMKkDghyWriHTk+xaMcT6Esu1w2wEizquM8ZkVFowX+XBAxgGxMMrS
Jn5tRUEt4PzKhACnUIeskgb3fgTmGISgaQxQYQpqBUOThRZo/MMtbou9+IQt
T4WK2HRJNuFSbKSYZ/IKWfRu0P7VQGCeAijqk1hNuzvci2eXV5dqqR+Z/hZQ
8GKUOhD+o7Vlm8QodTRwpaAjxzHyM0SfSd7EBWjhcQpzU1NqHXaCCSlwhQ8C
1LHFjb9BsbfGt0I7RhMkW2BX+K4V8uzMcsdRSozRw4kyLZh53Xh5u6EBRxHl
dLtPditZUqWczTCh3xZgvrA1R7JbyTC3Qczmli+Hr5NC/q2xgHZok/c/1cXi
IxyhXc1bn85vl3gyQp/WVkgsVyJ0IRnd/RRfbTtHbSOpbVvbAMyCHpaORE4j
IE5ZCboljj7RcBQVjWNEUiEaXebSVIBbE7vTkCX6evmM1w3DTOLIVkYmizSn
+Ef3UtCembqmRcjrSXeLaJaY7gtW3uXi2s+bE3yvXLzq+HQmul5fJJcCz7K0
vVUuP0oufJ/g/06lV7n8DLm4OPnU9l5xlctPkIsYhcNu4TDr9Y28cbnqy8/l
88tVX36MHVMXg/tWufymOb0fFAgeuV5HyZ/M7NTPkstjfwU7YFjMx9fbfvf0
tX/b7Z9/Lb10/8qLn3wtXvzCb335yEE/ek/dN031/Mpdf3z+zlXnV+778ysY
ygt/sjh/hdlYuk7eZQfPA2d7euEHH17+rf/nF7OXf3AnxrtOe/+CyOX+hd9v
/4qH2b9yLvsFEPNeuey9/+mqv+vF3/aDvfKFEKX5v/Zwv/QdX9kT8tIwcfbq
by0/67jkl8aT///YNXG7wPjFvzD7nIakL7uIi3A47fb8uV8SrLnz6fBw36iy
pBRhTgjRyyd/82B3OLJnj3azOwXPBZ8HhCocoT3/YrHggE762Vk3+OXOuXzc
kknZ83b0KXvuNzeYqn6unvyzix3j56d+cJHA6bcNGoRftKMfWUp885N5/hvo
E3/yfnBjr9Ldk54lI7kkT5qz+EwUiGfMoNLQF9xL5vCl6kJaooAI1M9N5QDb
6HTcP5eiuf0u5/JZ/xC+vR3F0/qi9f74uECJbYTk5Q1/+NtXeyeaTx7tkyd/
TPxW6LB9XCjdgAN50E8GROYciXP8LFC22Z+PmyftJ8Atj2QjvjoiIndWq+fu
Wvod4/PToWb8bIrGE/zQ5vhCTNXWT9nfCPpy+Go7JrrxKAbzyZ+TPztHvDk+
HXWcm2d/9eLExbHxnhyhGHnZPnru5541RIr2+i/NKyh5yHanXfv0d0j2u714
7n5Qkr89H9O0p9ORPZsc7g/7/Bl5iuPh7VRSnKy9X3zJC3xCXH/WUx5d6DzQ
h8W7563kT57W/wvk8hH6MG1fVOKi/eGrgKP1ugT0LtumRt6w3ZU09qaoD/RB
erMd5uv1ss5IeSGwbk/lOBPsjsYHuerMV0DVbilPv8/SG5ZVytnid7Fasv9T
NNaNSDUn44++ZbILXv44/PuPeZdLCE0x5/Blv6eSDaNbRwTVvC8XxdfjfVEy
fN6EI8rTmxkWW9o9lXkP2Bdpn9AkR/auqvRLzdfrSUQBB/8mZ3anMBeoWDIg
n007dL2de2riaNNUN9nFbns8Mu+1qtp6vXnMP+2wvfR42p8x5MBuPg3O50Pt
XAzIXCE/nnaJOL3tKyAIACSTvQPg+PGwr7x6dzo6LK7V7fm8a7zj23mb7Q5o
JF0t2hOX9SNvsjozVMAAcJoDbdZE7cQ77iPRbabNoRvB0csOovbtNOuqlYRL
ktDaDa1L2AE8riBfrzp7xyNP98yWUHK20Pm6Xnf6F/WG/jUIIiac0dYYD6iQ
lj70J3bdsDhjwuz3GfUfZkdqC6XH6LKd1SSkupUH26R4rkkDI3rohOGj3upn
nkWND71cIJNjbg2bJ9u2+8QVCqj2uI+oPHusJaqYfQXAyQWfQJXwshr6EnP5
ZnUMa4V28XrAL/iX3o4Jfahgdk7oQ/bPETtEVqWyvSI7hnAgOXoKWlW/Ob/v
dXYs2hmJksAhM+3RCvNNFGDBYkhi26ze5dkL/lq0nd8/nwpRAStvebC3ft9O
3NzvAnY8nDDbcLdvitPhuJcuRpOj34cmVYfz2WctJHsS3mFfhof9OXqNwPZP
27HJGio9kDzItHVfmSCQXT/9jKAj57xeNgVqBIE6Kybz4oXWhqab/cGCQK54
yg/KMzsIhiAxBod+2p3Oi4ZIoK36tNvyVW7/U+QsLutkfIWLfwiSqSajVtkH
CrFK6H8mjF9vP75FEFiv/7EcI9WqET8xeL7/aSti/D/IQ17MVPf6IdF/wKn8
B0DRcoonH1IHAAAAAElFTkSuQmCC
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 6</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see a clear trend wherein cells that have around 5% mito counts or higher also have far fewer total counts. These cells are low quality, will muddy our data, and are likely stressed or ruptured prior to encapsulation in a droplet. While 5% is quite a common cut-off, this is quite messy data, so just for kicks we’ll go more aggressive with a <code style="color: inherit">4.5%</code>.
<ul>
<li>In general, you must adapt all cut-offs to your data - metabolically active cells might have higher mitochondrial RNA in general, and you don’t want to lose a cell population because of a cut-off.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<h2 id="applying-the-thresholds">Applying the Thresholds</h2>
<p>It’s now time to apply these thresholds to our data! First, a reminder of how many cells and genes are in your object: <code class="language-plaintext highlighter-rouge">31178 cells</code> and <code class="language-plaintext highlighter-rouge">35734 genes</code>. Let’s see how that changes each time!</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>log1p_n_genes_by_counts</strong> &gt; <code style="color: inherit">5.7</code></li>
<li><strong>log1p_total_counts</strong> &gt; <code style="color: inherit">6.3</code></li>
<li><strong>pct_counts_mito</strong> &lt; <code style="color: inherit">4.5%</code></li>
</ul>
</li>
<li>Everyone else: Choose your own thresholds and compare results!</li>
</ul>
</blockquote>
<p>We will plot the raw data before applying any filters so that we can more clearly see the changes we will make.</p>


In [ ]:
# Raw
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-raw.png'
)

In [ ]:
genes_filtered_obj = adata[adata.obs['log1p_n_genes_by_counts'] >= 5.7]
genes_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_n_genes_by_counts'] <= 20.0]

# Violin - Filterbygenes
sc.pl.violin(
  genes_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbygenes.png'
)

In [ ]:
print(genes_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-1"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-6"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-6" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-7" style="max-width: 90%;"><img src="...
"" alt="Violinplot-filteronce. " width="2715" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-raw-filteredgenes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 7</strong>:</span> Raw vs 1st filter - genes/cell</figcaption></figure>
<ol>
<li>The only part that seems to change is the <code style="color: inherit">log1p_n_genes_by_counts</code>.  You can see a flatter bottom to the violin plot - this is the lower threshold set. Ideally, this would create a beautiful violin plot because there would be a clear population of low-gene number cells. Sadly not the case here, but still a reasonable filter.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">17,040 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
counts_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_total_counts'] >=  6.3]
counts_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['log1p_total_counts'] <= 20.0]

# Violin - Filterbycounts
sc.pl.violin(
  counts_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbycounts.png'
)

In [ ]:
print(counts_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-2"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-7"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-7" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-8" style="max-width: 90%;"><img src="...
"" alt="Violinplot-filtertwice. " width="2745" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-filteredgenesxfilteredcounts.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 8</strong>:</span> 1st filter vs 2nd filter - counts/cell</figcaption></figure>
<ol>
<li>We will focus on the <code style="color: inherit">log1p_total_counts</code> as that shows the biggest change. Similar to above, the bottom of the violin shape has flattered due to the threshold.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,678 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
mito_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['pct_counts_mito'] >=  0]
mito_filtered_obj = mito_filtered_obj[mito_filtered_obj.obs['pct_counts_mito'] <= 4.5]

# Violin - Filterbymito
sc.pl.violin(
  mito_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbymito.png'
)

In [ ]:
print(mito_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-3"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-8"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-8" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-9" style="max-width: 90%;"><img src="...
AAAA/nf/ADxHCLs5/RuAAAAAAElFTkSuQmCC
"" alt="Violinplot-filtermito. " width="2670" height="958" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-mitofilter.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 9</strong>:</span> Violin plots after filtering genes, counts, and mito content/cell</figcaption></figure>
<ol>
<li>If we carefully check the axes, we can see that the <code style="color: inherit">pct_counts_mito</code> has shrunk.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,605 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>
<p>Here’s a quick overall summary for easy visualisation if you fancy it.</p>
<figure id="figure-10" style="max-width: 90%;"><img src="...
"" alt="Filtering summary. " width="6592" height="4644" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violins2gether.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 10</strong>:</span> Filtering summary</figcaption></figure>
<p>Fantastic work! However, you’ve now removed a whole heap of cells, and since the captured genes are sporadic (i.e. a small percentage of the overall transcriptome per cell) this means there are a number of genes in your matrix that are currently not in any of the remaining cells. Genes that do not appear in any cell, or even in only 1 or 2 cells, will make some analytical tools break and overall will not be biologically informative. So let’s remove them! Note that <code class="language-plaintext highlighter-rouge">3</code> is not necessarily the best number, rather it is a fairly conservative threshold. You could go as high as 10 or more.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-1"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-1" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li><strong>min_cells</strong> = <code style="color: inherit">3</code></li>
<li>Everyone else: Choose your own thresholds and compare results! Note if you go less than 3 (or even remove this step entirely), future tools are likely to fail due to empty gene data.</li>
</ul>
</blockquote>


In [ ]:
filtered_obj = mito_filtered_obj.copy()

sc.pp.filter_genes(filtered_obj, min_cells=3)
sc.pp.filter_genes(filtered_obj, max_cells=1000000000)

print(filtered_obj)

<p>In practice, you’ll likely choose your thresholds then set up all these filters to run without checking plots in between each one. But it’s nice to see how they work!</p>
<p>Using the final <code class="language-plaintext highlighter-rouge">filtered_object</code>, we can summarise the results of our filtering:</p>
<table>
<thead>
<tr>
<th> </th>
<th>Description</th>
<th>Genes</th>
</tr>
</thead>
<tbody>
<tr>
<td>Raw</td>
<td>31178</td>
<td>35734</td>
</tr>
<tr>
<td>Filter genes/cell</td>
<td>17040</td>
<td>35734</td>
</tr>
<tr>
<td>Filter counts/cell</td>
<td>8678</td>
<td>35734</td>
</tr>
<tr>
<td>Filter mito/cell</td>
<td>8605</td>
<td>35734</td>
</tr>
<tr>
<td>Filter cells/gene</td>
<td>8605</td>
<td>15395</td>
</tr>
</tbody>
</table>
<p>{% icon congratulations %} Congratulations! You have filtered your object! Now it should be a lot easier to analyse.</p>
<h1 id="processing">Processing</h1>
<p>So currently, you have a matrix that is 8605 cells by 15395 genes. This is still quite big data. We have two issues here - firstly, you already know there are differences in how many transcripts and genes have been counted per cell. This technical variable can obscure biological differences. Secondly, we like to plot things on x/y plots, so for instance Gapdh could be on one axis, and Actin can be on another, and you plot cells on that 2-dimensional axis based on how many of each transcript they possess. While that would be fine, adding in a 3rd dimension (or, indeed, in this case, 15393 more dimensions), is a bit trickier! So our next steps are to transform our big data object into something that is easy to analyse and easy to visualise.</p>


In [ ]:
output_h5ad = filtered_obj.copy()
sc.pp.normalize_total(output_h5ad)

<p>Normalisation helps reduce the differences between gene and UMI counts by fitting total counts to 10,000 per cell. The inherent log-transform (by log(count+1)) aligns the gene expression level better with a normal distribution. This is fairly standard to prepare for any future dimensionality reduction.</p>
<p>Now we need to look at reducing our gene dimensions. We have loads of genes, but not all of them are different from cell to cell. For instance, housekeeping genes are defined as not changing much from cell to cell, so we could remove these from our data to simplify the dataset. We will flag genes that vary across the cells for future analysis.</p>


In [ ]:
output_h5ad = sc.pp.log1p(output_h5ad, copy=True)  # below function requires log scaled data
sc.pp.highly_variable_genes(output_h5ad)

<p>Next up, we’re going to scale our data so that all genes have the same variance and a zero mean. This is important to set up our data for further dimensionality reduction. It also helps negate sequencing depth differences between samples, since the gene levels across the cells become comparable. Note, that the differences from scaling etc. are not the values you have at the end - i.e. if your cell has average GAPDH levels, it will not appear as a ‘0’ when you calculate gene differences between clusters.</p>


In [ ]:
scaled_data = sc.pp.scale(output_h5ad, max_value=10.0, copy=True)

<p>{% icon congratulations %} Congratulations! You have processed your object!</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment</div>
<p>At this point, we might want to remove or regress out the effects of unwanted variation on our data. A common example of this is the cell cycle, which can affect which genes are expressed and how much material is present in our cells. If you’re interested in learning how to do this, then you can move over to the <a href="{% link topics/single-cell/tutorials/scrna-case_cell-cycle/tutorial.md %}">Removing the Effects of the Cell Cycle</a> tutorial now – then return here to complete your analysis.</p>
</blockquote>
<h1 id="preparing-coordinates">Preparing coordinates</h1>
<p>We still have too many dimensions. Transcript changes are not usually singular - which is to say, genes were in pathways and in groups. It would be easier to analyse our data if we could more easily group these changes.</p>
<h2 id="principal-components">Principal components</h2>
<p>Principal components are calculated from highly dimensional data to find the most spread in the dataset. So in our, <code class="language-plaintext highlighter-rouge">1982</code> highly variable gene dimensions, there will be one line (axis) that yields the most spread and variation across the cells. That will be our first principal component. We can calculate the first <code class="language-plaintext highlighter-rouge">x</code> principal components in our data to drastically reduce the number of dimensions.</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment-1982"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment: 1982???</div>
<p>Where did the <code style="color: inherit">1982</code> come from?</p>
<p>The quickest way to figure out how many highly variable genes you have, in my opinion, is to re-run <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with the added parameter <code class="language-plaintext highlighter-rouge">subset=True</code>, therefore: <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes(output_h5ad, subset=True)</code>. This subsetting removes any nonvariable genes.</p>
<p>Then you can <code class="language-plaintext highlighter-rouge">print(output_h5ad)</code> and you’ll see only 1982 genes. The following processing steps will use only the highly variable genes for their calculations, but depend on keeping all genes in the object. Thus, please use the original output of your <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with far more than 1982 genes!, currently stored as <code class="language-plaintext highlighter-rouge">scaled_data</code>.</p>
</blockquote>
<blockquote class="warning" style="border: 2px solid #de8875; margin: 1em 0.2em">
<div class="box-title warning-title" id="warning-check-your-anndata-object"><i class="fas fa-exclamation-triangle" aria-hidden="true" ></i> Warning: Check your AnnData object!</div>
<p>Run <code class="language-plaintext highlighter-rouge">print(scaled_data)</code>
Your AnnData object should have far more than 1982 genes in it (if you followed our settings and tool versions, you’d have a matrix 8605 × 15395 (cells x genes). Make sure to use that AnnData object output from FindVariableGenes, rather than the 1982 from your testing in the section above labelled ‘1982’.</p>
</blockquote>


In [ ]:
pca_components = sc.tl.pca(scaled_data, n_comps=50, copy=True)

<p>Why 50 principal components you ask? Well, we’re pretty confident 50 is an over-estimate. Let’s visualise the variance of each principal component.</p>


In [ ]:
sc.pl.pca_variance_ratio(pca_components, n_pcs=50, save='-variance-ratio.png')

<figure id="figure-11" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAYcAAAEbCAMAAAAGZLh0AAAAYFBMVEX////x
8fGdnZ1+fn77+/v39/dAQEAyMjL+/v42NjZISEgrKyshISHk5OSSkpKHh4dS
UlJ3d3cFBQVdXV0VFRVubm7r6+vc3NzGxsZlZWXMzMyxsbG5ubnU1NTAwMCo
qKhLD+CPAAAACXBIWXMAAA7zAAAO8wEcU5k6AAAfAUlEQVR42u1dCWOjLBNG
BQS871v//7/8nsEkTbvdIzaN++4H3W2qEmPmYS6YGRj7rzaubn8qpv78fYJx
5tpTm9rJzy//XTsDg9vwfpCN7C8H2zegwdkjcsmxzjfoCM64YOohIDj9c1g8
XVWLhzlIsccYyLXfEBSj2vOIKR6iKtindeR7Lg5iDIW1RB9Cwst9R79vaN6V
O97kjXrvVXhXG5cV1d5fOZX9BXLvv2/ka8WFmB6pCW6VhbrBcdMeeEt7BSJr
EnS+OnNOTRxofo5fQ+b5uoxSxqawMoMfK5bFpfaZ8k0VRxtnSVXXBgSXdb7t
aEXGRKqJa72xuSyjfKxSxYY4yp2eONKWEmM+7tm8iLUe2VSmS9vFAGQRW92x
tJy6FIem6UTPJrMmm0zsiC+bpWVTB8BaFjRgIumzzIzdUDmiHrBSmd7UGnEr
n0zG1jIBBrEVSUm4sQ1U9souKRfqHYDGSbzad5bTLr6SqLf6gYF98oCxvnZU
PdJSzaqGsy0v4zJgM1Fxxa8sKnW5iRTCiJfJWJI2YLKOy7jerH4oO6CQhei1
sqxilh8sRKUTTEdmMNq6h/xpa58JDOc1BnXXWIwgaatTtmlL8q5OqH8TMHb1
E4ADxj561TMLSMnIlP6xpXY4HGoyj2AClTN0Q0GsoIgpOggiv/SZHxLJF6aH
js2sBzpJ791wmGvRpnhnKsEi4Ac/7rzK/DfdJs5ONvbSOIDvlsa1qQo265Zx
8AMLyriRvvJh/giQPGnqchBskmU9LKQXrJoQQx0PemKJLE1XZUxkdd10fwFh
xaP0VOzMKeP9Y5PbXDe/HPC366Sx28/mVLvru1p7yds9EaX+sin8PwburzKe
+OWRhNj/4LdH5O8W6yzRvevTczs3y/d3sb+CHR7z6/k/43uqv88gf+iZ1Pn0
Ux+Hjvg9z6r7h//7JpQEf/CpvOsfyX9s+fovX4l7fHhbHSi82fu7ePrXX0/9
5WIJmuvBJ2rjOkaLylDj5/VN2w/Vl9dQR/RvfxYd6cszvT3apdf1kr6e0ne/
w1O+yLtWe9ZueCTsR5bb5m9+5PzJJ7bwcTEWFQlZTalw1DsPB5oda68SzZHv
PBwYi4MR7tMaOOKdi0MkPdIooYsQPRUHTxqiP88d8c7lBz2kMxsLJ5dOxiFk
aV02m6Pd2TjIbQoGF6x7ulyqMJGfey7m52Qcah1kQRyknqPeqTjIrZ/neZ0c
8U7GQfxjK0L/VT/uTkE7PX0yDpc0DscSZ/oPzOWY/Q12q6Oaw8Hh4HBwODgc
HA4OB4eDw8Hh4HBwODgcHA4OB4eDaw4Hh4NrDgeHg2uP4HC/3vlZLo36BAe3
KPcN/ODrqKHXNtcGqfiypqNJR9UtLiD8D+Re/udx8PToyR784DftNqDgWlDg
ZDV5w/ZeLt1Iz10axDfgMErUTKNUHzOhHgiStiZUdlliKuB1e4tyDPHtOBAP
bCA9ixLBYqTbpzjRGcX6SzEQz5PXGs42dV/9bSUG/g0cptxigawfhE1SPRy/
YWKBKOqlvR6YobwrzPxWS8G1p+Kw5pwFmWflUkJYbBVVagEexT7qvUS+VRG+
1oh07el2a9ypaKF00IbNlPLjN2Sq9iKfrrGUF7u1K4qOqdSlK36P3TpFceql
G8oXxRJli+K6RDW7NYyL9oMfZ9Y5T5lxKHyTP60uRTSUUh98Ov4OB6lYm1bh
XcUN175rXkPcFzPn7/iBVzCokmkvbedUxPNxuNaOUj+rJKL3q9eyws5YesE8
n/rpPJ8KRlxcC6cfXjTf+kMlcn0ffi8dCV82763eV5S62kvc4fA3rD9kzSK6
IXMkPBkHQdnsk3LTfOfiICrfzwY37X06P2iYrEYJl551Mg5RVgR1lrn5pZNx
mNe5x8/ODU5BnBmvIa6b6Dk6noiD2gudO1Y4mx/UbULK5bO7ODKHg2sOB4eD
w8Hh4HBwODgcHA4OB4eDw8Hh4HBwODgcHA6uORwcDq45HP56HO73ueTss2CM
0FHtBXm7dm3tGsWtPovDd6GUr+CHNNSXPPZQ94ierEMkTgc6Gm7hANoFynw/
DonsuhzBMMoP2qliS9h2OmmjlplrHrtw+uEFOFCKbpGS9JmYqEWGOGI9drEn
8lX8Io/dtSfj4A+UP027iArKYx+2lg0+C+pYXhLhPG4c1V5QT6DZ/7MYmzHF
SQPWyNelStohtYo7laZ0VPt+HKhsQEBJDfmK+hqM5FI+UyJ1igwsZQ0opx9e
gIOIFh51u4DyB75EQKYdI09VvvPjXlp/KYwzFaDYTBWj/g8LohACqQijof2Q
P+3a989rXB1pz/4T79PkHA6vwuH9fIbn5pdOwUERBuLjrIZyOLwUB3GVQz9B
weHwQv2g7ks5WP54+9vh8Cocfj5vwR0/uHUgh4PDweHgcHA4OBwcDg4Hh4PD
weHgcHDN4eBwcM3h4HBwzeHgcHDN4fD/gsNY+GuVd46S5+LAzbrpcXE5EWfz
A8qtS+5k1dk4eNivJmGtiwE/mx/ErXKla2fi0PTAYBwcJU/GQVIwvnBy6XQc
nE/xffGtv8PhLqc6bRbV7clCrj03j13dCC1+/5YW23FUvUtdfPp+7FSwW1gg
uAXidvjpW/btOBwM37Afe1jQa1eF2OqVZTHlsbd5jDzez/VDi23Cndn69Dx2
syRmxfhPi24a+Ki9JfJYlbJWfB7vrYMG23EEjpJP3o8dJxpKScztvsc+8kXN
srxLXQ/5u0Tfce62fnWUfPI+4ANtNY1UrHDBHsgeZYlWWxdVeuh2tS08+Unq
ilMQz85jp3oCFaP92Dny2Asoh2rqy5E1md18PQ0/5rG77Xa/JY/dUH0NDG8z
X/PY5Yo/2GrcOtALcVDx4umR/DPUm9nz2CPBsCN4VlyFj8PhVfux8wJ57DIM
bf2lCAbrKqM8cfxwwvyS936OQ9wpAofDi/J2Pz/gjh/OqetAKFzL9NndyZx+
eCUOYucCtVMeP94bLwiHw4v1w62mgLrfKc4C4nBwcWQOB4eDw8Hh4HBwODgc
HA4OB4eDw8Hh4HBwODgcXHM4OBxcczg4HFxzODgcXDsDBxddeToOS+ct/dA7
gp6MQxzlRVBmjqBn80NarF7jBNP5+mHZqsjR82wcKJcuWRw9z8ahoIyguXCC
6Vlx9xjZ6gAOkrktkZ+cP634/TZBP7oGn+NA0eDC4fA0HK4E99RHAMSvcAjS
0esDV1bgiXnslvY0vL0PRQYugeCf20tZXTaTUw9P44cN+1rSa2s07WuZRdFM
nBBW3i/5wfhbUDlyPg0HTy+dhO3DkbyL3NEx4mPYQuzk+a/1Q45OrvjPk/c9
TiF+DOWx89Tmi7Iln/Jfy6Xab4q4cWUFnprHPtj8acXitrH506xae/lrfuin
fu1XV1bgqfuxWxxQUCNuB9QTGLapYXO+q4cf89ivZpXT0U+tr4FxX1B9DdSX
6WpBcsn0RRnFZdbuFRF/VgPOu1q5wpUYeIKejrpWw05Su55eQi+JiKxz1d7e
In56t5V5jWlcceNn1F+K4oClG+Wxk90UhBFxB3AgJe39JsVUokxN2zu76Rl+
3E5vdZU04pK/6/12n1dclMJwe0+nLJ4z36ruKsGJD7Me+qcweEYaoVrj9MNz
5pf4u4qUHwEJf74r8g5A4sj6TXVD1d1/egv/CRcVM6mSjDu6nhu/FLoAp5Nx
2CsPWGePit87PX0qP6RN13aFW4U4FwfFvK0saUrKtVP5gVfZHAw/VeOuvUA/
2HJB2vrUwgFxHj9YzRxnRVZnmdPSJ8olMpVWrEH0s1uF+AvyHwS7jylw7bw8
FHVdj3BAnJsPxDe2GFNNjron52VJZjrWuWWIs3HIO1Q/TlyQ5dk4LIWsdTU7
6p6dL4q9LpXbq+Z0HAYqmL9GqeewOBWH0HoPZWpkmjgoTsPBq2h9lGPzjsQ3
eeuIfAIONq4vCBZvaSK+qwqX4H4GP9AeyMIv62q+bt/k3OpT+AHNZFPR8GBP
XXQwnKUfKAJWVJf9YCPHEefgAHGERIi0zGpLfsOcxXSW3YpViLlfmwI7IjeZ
09NfweEuTZQfrL9EOyI3s+WQ9t0WQ+8C8/mNX9RPIzsfakqpfwQH/hZf/G47
lMsff4hDO2x+iijxqWqmJpx2cndDLmXRsyut1IXcPyQJf4GF1D/CD2LfI4jv
45GLo/XIZMJEzjyZLPFy+RzPLLjdkv9Ib/5/Xlcr/Cyth1+2AOdvG2bdHfwp
DlEWBHWTauUhc0XvNwnp9othb/txCbdb7Gc4CPYW2K3uqjvchYD/KQ4bKet+
zmRV5DLdE+NXmeem2uy9hN+q2+5c3v95dblP8oEiux+7l8fY+hh57DplYx7r
4EahR8OIQehELJmOcx8ZW54Hpe3bVgc+XL10afx5sJHiF2b8SlPq38Ch1V1L
dQRYFoipQn5cssjE67HK2R+sV1l1nCbB+3apyLkmOgW1v/n+Vq5+tSlt1jmc
12r0ktVOmH8JCs7VP7PvcUA2v9332OaLSkscuR7EwVjBg2TfxDAJ8vupHw3W
kI3BAE0jtb1lKauqKZtVtYUslo2NQ1iNh3BQ/0Yeu2J+Y/PYkT+d2Dx2Chxe
Iu8oDnukn8n1zIp62zZ/i9llGRsD3y8xS+5hLqQKepYjxEDOySglbbWc1L5N
LVrzAfEfPhZbq4wvuTbRFaEqzzE8mtUEvZbpjBA2PV12GyTO8+73IvyZIabu
jOaLPSLsPbzPZwLEnR+0i1L+S72mfnIsfr8f+0DJhgvjsSAMaObU0/5+PZWf
57H/vAXZyLsqbTl4oC2IsF66P0tx9enoEAEe2RAxk4BXPJMnyP5N6owUfJsv
Xd229egt+VrJDt0XWWRNEWRl5+UFkzCNa9HWpsmK2oc6q7Qp2JbHQyqznj4D
yyAz/Jamjk2TtIM0suis6ZG1RQAtGHTDtg75lCxdP4wz6wKDeOm+kmbMUzt3
P8pqzPXW6LjKhKiklHgfnltsUxZgVGR3pp/3lkJFaHLvzRbd3SX1o5f6aR47
JTDkyGOPPR/qWUJCmVtFUI+bx5S0QkWgYW06PM2U7sapIvWQ+jagw9taOwjp
M9uR6Vy3rKvGIY51VZNTLk3IRMwZDrqKSUTkCJboeBoRxFnibn4V44kqwcKs
mZAJY1Tei7YUVcK6PJqGaOjavE3ihJUJ6/M21xhhbNYpnmGK5LrlPjN69PW6
1HrIs7LRrPG9KUbwj2i1RgWFtGMR7+qElwtbh7XSVIiwj1N/S/04X7chZWHh
r1XVJ4vX9dWyYsSZKgUn62Eb/ATffc6bPg/npAgj3K+DOxsN8y/s1nAU+37s
RYf92MdQLFjPqZqLR2Hf8pAAbnOQfCBhhoeJih4jZCzq1Oppf8I1HcwpDrUd
NZcUFvqeCRhhf6RCGhBD5xJjs1iHMNf5ElDZNA5QMCKlLTjUVszzq5DphT4r
7vIRDiPN+faVDlkSCQ9QeMDJgCeFV5LFlmqJSZeigRykmnZ1NnTJwIxHb4sp
+IfLkD5BS6kS+D6Yumxzntvia5CafrptMbJjl7Sp5zkcl7w0ZmjKIIQwb/u6
rRaMhTg1ebCYpI9aT8vFG6uxCROMoM7wX9itOspYAGbIQ5ScYUEc+mwqowiC
d5eE4iH9wJnk5FbnxLY8t+8VuqEvojCggEYc2ZdyAodYzQFfwm/tEzZv80we
8Gu9O66/m/ZK9gdrSQokLG+qloswMHXdzMMuITY5+FIPRspe+T6kSpXv2ZZB
nuB3WucgJrFbGwwhkzkNirKQpZSrsfKmzfQQSAyFSvqs2TAk8GONjQC0poUv
O8OvuyZYRNMaqpAnas+GcSERIdnyEkzdMm2I24XGWBBwB345r+Fd9j4mw4N/
skm7fmwiOya3OtDZqpImkHRnMezeYiWsEVtYOmJ0gUdCH7wxpfFFcNmPT3dP
b5eMQZcVI+R6V2Tg2qaAnBf+UgSJYkVDp4KLvm3VmyoU9F2uPmnHUMElsWUI
7Qfz/Wt6lyps3rj3I9xVu99MXLKdoLmTfdZHJTcBsd+3ogFOCy9dMGgFqeNx
EVVRHBc+VcTgKgurzMgszUleFSOGhP6VXHpvv4v32p4fKUq59tM4r2sWl/nM
smLhSzFY7dlW7G6XcRtbINLan4gtSPaCR0h6pRGdIoZJIQdCOc0Q6Jrk+qbq
eTUbdMo669WL16mCqF+s5h2KdGUio/GJu45Bhu88ZA2GoT82/joMW5D2Vd4l
1baYvOuqaTWNn2WdqcakSZe82optzfOkp97o0PhjXk2Nv+DcmCxJX01Jt3Qm
6cauHc1y2xYaavBtfhlMRihdNDUxdAuWvQXE8z9df1DvzSt+ZMliXypNp6Ch
MmdlOUz7Lby4twunDXkmfbF3HuwjZ0OyF9ciiRXF6QQ4YiiRzY8wCrxgIIHO
h6ImD6TQNDKGhg4K6JKZgIK8hjbRZgfK2COgJgnCforHpZ7WuO+kHrdoTKTu
N90v8TTHY2fkig5dPU/R2Emz0gudi/CmfsabdG3I16nDYajivIbjM8RAqktM
R9BA/rcdLC/Td0uy5AANR2YGdp3pqBvAWzkKwgzJ6/d/MDRTIliVTtlwvUW9
s1dosa4tGs07b7qxQyegF6a0HWRZTiPPL/POvtAVv7Z60y/pSlqT4G8IFEb6
V5DI14bOFaW9ctHJpf2NA47IHrx48HMu5/AXOZZWzFBvOr4ocvtbJ8GwwMJu
sAIvIYgaHFg06oG80aJsdqe0tOcqXBqqurx0sBA2EopczIa/HIe4KII4SCOC
A3YN8nuXwC6coiaHZYF9FVU2PQzusSrIZBuHey/oVjNT3JWyvkON73phCXag
LE61IVHgx5rwSmNrB9aQ1/AcYbFAUxgo6IReFF4SQM0Fl1Z9tnBUWlSkhUEM
T8ZAbOIFmhY/FTRLAF+HLVlOpITjAwVdLF4l2iYYIW+7ohnZ0Af4GrmyB2YJ
sDxZcYC3tIOSgub+9ev5YR2nceonXRRYuc7arawbP81Q+S8rUjx60sRWCEp9
5RGicU2Cax2aFV92kqTVVivFxNo0VCNqljNcor6xkm3V1GGyc4nWxrhNFgux
o9few7jcWR7Xc939uY7dLYddNHJ3uYLnaRe6nWdne1QykoJuiJh92mCan3fp
EBNQBJfqMtjTHB0iqoKe4buaHJ1a6Z21L8080lz4lGcrVRkNYlIWIqvrfM4a
eF9NallV7PatqvdiQnvgx+7Gh7mlEMkJ73pO75ZDaSkdFpajCDVvzWhnENT3
DXCkRtnD7uiLvRS5RjdvbJoRRKMXDIJRjzQlWdDcrxjDBXq0LwZcafuG7tqO
KNIGABq6negi9BY9BgZ5aQP5JBYUjnLzpJoFLXnxdqGHEh4deO1iEUS/PfKX
n7Q/kLhaCWRdN23l+xkcRLOleIEQIVACYg7wSDGqsbDzUBe3idX2tyQvjCW5
FFbT72E6uyV/kWy7i1JaY4+kOzpGltm8HTXS9DiqrdlKHdr9hZDc60vtNCmt
sSiNvQMxqHe9Q7ibGbW43o481LXdWVe8IalHfALCJUa6FPatPZhh/PZDQAbj
HJy9T1MUZEGZ1rsRDBHNkUFk3W7LI5jUEmlcBhNEagtuztKVJ5VBsGY3pEE6
sbHIAvgjXZMPnYcOabaIZDBFrzCRZayNF1sfYAelrXI7c2e1BPxpS7+YZhdb
miYhn3+fI4gsuLnt4O2nQkNwtMZ4VqmQUyd2D84L7YRMbu+gSquczL7seJGu
O2ryxq2QoWa/Aqj5e+K/HAc75OaVBFSdke4u6qAo4iyGGwGnG3CQfZunWwp1
iPjMfGNql180T4WpLr+uKcWF/JEeeqakPLygJucktR0ChCEsA03DzVmWNj0f
kdRdJHBbwmzh3ZBlcHm7Jkx7tWA4ZD3v0Q8adQniFHgWAcJzIe1jeIVYoiLn
kPphBI+BbBKvw/xigd2Q0Jv0Whakq1iKmJBujSRu50RtYf1ngGiknfi1uFsR
Cg/RmI+e2Ctx+DgvtdrWz/QzxwEQyQqpdttxz7LL4F1ggML3GGzlajBMlYG5
LVBBgQPrleT+VBTkohSNtZUIDvwuNppwIAjZ/pICtdn6MACPYnBxBecwPSjS
eu8ATXXrbU9Rh2Gz56qZb/ub7MHlDtz2g4NKbJrCNOrxMnQQq9I6rfbcMhhY
tz04F/PI6AePsk2G7Gs4fMdCi1gACdzuqMkqWRRRAOsjiLlFhFYwDE3zKjAM
RIyHF0nsg3krRudJueceLEF7kE4pVpzg8+UUaTsjdkeZdE5zpn2LGi5BAobb
hlOCDmy/yTfUIat2n3NAt4muoIM9Z/vJ9NJhDvLLfehKljPLuj0scouQR3j1
yc6t3QQkV8+33Cro3KhofFT9nY/86JY+gn/XZhuXOYF1XPF/Wtdxm/epKZJd
QZ2VGf2m47tz+8F+rqRTRQbhnTSUgtEN1n1MQDgA1zWi2muQYyooyWkeJcE8
F5y4S79u8AZyDqxjhzlX9MZ7bG84+fQCy8Ij94fTWGha9MZ96K7dICRgpdsB
FNwup3ovogIHY+sqvEDGVnRFkdzNbD+f+r3fx+TB8R2/bLFws7Wqe2vqjuk4
j9vYT8u0bCOFgsAjSenqauNCpoU6zOsWBjDQixImV1wQQlndUM0PVB8HSMAO
+ggv1CGIccX2o3PUIciymsQjcC5KOmWdT3sj6tfA9bG4Q62gd5PiABdxJYgI
NcgK6P7ccjAAxdyqMkBtoXPSQkjn2mHvN9yXkzywxVX0pOGv/lir8HfaRfxQ
NvCaZ6F2GwAe4+jT755kHQwCYDUvQA1qCKfmBfiuKdAj3PqtpyWleSZYUaV8
XvGGCZ0y8jyBO8CfF7wRl5bMdiToVxoEE91wJv+0JyBrQg0QNkGdEsJlRiMg
iCySYOcUgyC+9QP2Kb89tX48Yki+qiyouBdZ6s96i9skB//BQBCffldxX4ri
3Xnx7jFuFW2vd/beO6jTMtMSIVi2H2fgOxPuK8bA1G/ALcUrAJ+oH4ENvu6n
d0LmwSiVNjZhGOpSah3qKDzYtCx1hBvcjvXHPyIUVEYP2wWvEbrvn0bvo364
A26AI/pH/ffeEX60DG/3xrv2DvYM3haFMtzvKUuDK3dfQmv7Uftfe/9Q73eg
T4ouX9k+uD0Ia9zLvl/H9gGpFz36/rSSHkZqOsYB9cDHy/0e+HqxfKND+xgO
/FrmuPpicgl/5Ab80wiZivM/6/oDE3iXc7n4RWf1ybkf7i75H3z8rxj4Qgfv
0VBG9SwtEf+hr8E/nrlphPj9c38ekPLLe0dfDCAXb3fgb2HAPyLCf5BjP9Dh
aN6//mrgXXgskFU9ybFXB72n+7wN/pU7iK99iysd/K/qYP/w5yuuLnf4apLE
V76E95VvYTfp25HIzgyt5l+h3x8LoN+OR/GlL+Cx96F9j7vE1/0FXt/U22AQ
R3ZO+Zi2kjzNOD7wTo9dYzuOUOJa1vMr+8dssS4Ov7ltIkx5j0ab5WBoPeLs
IoROJgidHA8+xJIjDhNhuzI/vH0LQpOw4BSZo2NBkCGrBCzZ5NCIxlqubJf8
eFnKcYO5V/iLf2h/XnKwW6w7yBXxjEHwiG/+jopYfegR0hocVDOcZZXBcv+S
HdERFJbOIxoCTdqmR8f0qEHH7Lh6aWGsIcpxiY4qWNLUegyxdB8el66JnLGm
Nh2tXzdW/sCxajfJg4pWhcQHMSKsj07Z2V3mvlCxIYlsDAotQB+2dbq4La+L
XofUZFQPtLCaHDU9q3WUuAECX4+Ox1Ji86V6L81zhAicovP9Rh012JBmBBzU
NYLp0EMkcmNfwoFhA50Z46E7iMNW8N4QY7eHXdqEt+FE3mh9kJIL1oKK7Kh6
EpYfkGCxRIf5waugGCIKKVIHgcB4qPyyPSqXEGUSY7n2ItjEQdOXF36NBbr4
oO3chm2X94ctb9VrPEHq+Y/rJ25jk5k3NBSQf9PTB4wuCrzeSE8f365iq5Co
k2T+UduXAkamwG/9TByUzlutm6N2Mxa2orKeOhPJ5eAX8JYy1ohBlTGyHMQh
LblitrPh7XG7VdnEqUVqm09x5A5rHsWZSvRxOnzFkSYfTn1YLnjI4LtuU35L
T/tCZiL/2h6cfA8S5OKwD6l2q1udMTdxTQ48WsrEQiGuaYXi8LOor3wP/gyX
3E7fnzKx9L5uyoEhpO6WSp8wN8NP/Tbv58te/vhfrhjwF6RIqzt2cPXWfhAO
6hcUc+R6tby71q4RNsHRtROWNG62h8jI0Uy2lYuvL1a49gBD3Aox7NPIWUBn
u1v8i9sF+DVtiXOTR5IiWAcZ9lQWIa28YEqiKo8Q/ptLE7utgL9fTS/1KpCO
jrzychJpI7KU0k5RnyKeEn9QQZqMpSs2/v2NZvH92JQbQz3AXrIiooBh5E/U
C5Uz0pgPK51s+n41jTlorElghUzh9yhVVmGJRaFyEVJAAAtOstrJpe9vCXCI
AIVPqXSY0UVFASx4IYekJFgYKvrOTi69Qi4Z5SGxX/qU+owCP7BbZ90OE6dD
5JsbXcVOLr2gtVdPLrkLzebXvGm8zG6n+Jc5EeIuN13c13tbsFjhthZ8CQjq
N5ORbqvN7weBsw9FEflfNEf8fzW7dD8fzn8y/eSweNWa6y/J7Sb7XHPNNdce
bP8DuKiyMkFl04QAAAAASUVORK5CYII=
"" alt="Variance ratio. " width="391" height="283" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/pca-variance.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 11</strong>:</span> Variance ratio</figcaption></figure>
<p>We can see that there is really not much variation explained past component 19. So we might save ourselves a great deal of time and muddied data by focusing on the top <code class="language-plaintext highlighter-rouge">20</code> PCs.</p>
<h2 id="neighborhood-graph">Neighborhood graph</h2>
<p>We’re still looking at around 20 dimensions at this point. We need to identify how similar a cell is to another cell, across every cell across these dimensions. For this, we will use the k-nearest neighbor (kNN) graph, to identify which cells are close together and which are not. The kNN graph plots connections between cells if their distance (when plotted in this 20 dimensional space!) is amonst the k-th smallest distances from that cell to other cells. This will be crucial for identifying clusters, and is necessary for plotting a UMAP. From <a href="https://github.com/lmcinnes/umap">UMAP developers</a>: “Larger neighbor values will result in more global structure being preserved at the loss of detailed local structure. In general this parameter should often be in the range 5 to 50, with a choice of 10 to 15 being a sensible default”.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-2"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-2" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Number of PCs to use</strong> = <code style="color: inherit">20</code></li>
<li><strong>Maximum number of neighbours used</strong> = <code style="color: inherit">15</code></li>
</ul>
</li>
<li>Everyone else: Use the PC variance plot to pick your own PC number, and choose your own neighbour maximum as well!</li>
</ul>
</blockquote>


In [ ]:
neighbours = sc.pp.neighbors(pca_components, n_neighbors=15, use_rep='X_pca', n_pcs=20, copy=True)

<h2 id="dimensionality-reduction-for-visualisation">Dimensionality reduction for visualisation</h2>
<p>Two major visualisations for this data are tSNE and UMAP. We must calculate the coordinates for both prior to visualisation. For tSNE, the parameter <a href="https://www.nature.com/articles/s41467-019-13056-x">perplexity</a> can be changed to best represent the data, while for UMAP the main change would be to change the kNN graph above itself, by changing the <b>neighbours.</b></p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-3"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-3" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Perplexity</strong> = <code style="color: inherit">30</code></li>
</ul>
</li>
<li>Everyone else: Choose your own perplexity, between 5 and 50!</li>
</ul>
</blockquote>


In [ ]:
tsne_components = sc.tl.tsne(neighbours, use_rep='X_pca', perplexity=30, copy=True)

In [ ]:
umap_components = sc.tl.umap(tsne_components, copy=True)

<p>{% icon congratulations %} Congratulations! You have prepared your object and created neighborhood coordinates. We can now use those to call some clusters!</p>
<h1 id="cell-clusters--gene-markers">Cell clusters &amp; gene markers</h1>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-4"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<p>Let’s take a step back here. What is it, exactly, that you are trying to get from your data? What do you want to visualise, and what information do you need from your data to gain insight?</p>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-9"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-9" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Really we need two things - firstly, we need to make sure our experiment was set up well. This is to say, our biological replicates should overlap and our variables should, ideally, show some difference. Secondly, we want insight - we want to know which cell types are in our data, which genes drive those cell types, and in this case, how they might be affected by our biological variable of growth restriction. How does this affect the developing cells, and what genes drive this? So let’s add in information about cell clusters and gene markers!</p>
</details>
</blockquote>
<p>Finally, let’s identify clusters! Unfortunately, it’s not as majestic as biologists often think - the maths doesn’t necessarily identify true cell clusters. Every algorithm for identifying cell clusters falls short of a biologist knowing their data, knowing what cells should be there, and proving it in the lab. Sigh. So, we’re going to make the best of it as a starting point and see what happens! We will define clusters from the kNN graph, based on how many connections cells have with one another. Roughly, this will depend on a resolution parameter for how granular you want to be.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-4"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-4" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Oh yes, yet another decision! Single cell analysis is sadly not straight forward.</p>
<ul>
<li>Control
<ul>
<li><strong>Resolution, high value for more and smaller clusters</strong> = <code style="color: inherit">0.6</code></li>
</ul>
</li>
<li>Everyone else: Pick your own number. If it helps, this sample should have a lot of very similar cells in it. It contains developing T-cells, so you aren’t expecting massive differences between cells, like you would in, say, an entire embryo, with all sorts of unrelated cell types.</li>
</ul>
</blockquote>


In [ ]:
# Find Clusters
clusters = sc.tl.louvain(umap_components, resolution=0.6, copy=True)

<p>Nearly plotting time! But one final piece is to add in SOME gene information. Let’s focus on genes driving the clusters.</p>
<h1 id="findmarkers">FindMarkers</h1>


In [ ]:
markers_cluster = sc.tl.rank_genes_groups(clusters, groupby="louvain", method='t-test_overestim_var', n_genes=50, copy=True)

<p>But we are also interested in differences across genotype, so let’s also check that (note that in this case, it’s turning it almost into bulk RNA-seq, because you’re comparing all cells of a certain genotype against all cells of the other)</p>


In [ ]:
markers_genotype = sc.tl.rank_genes_groups(markers_cluster, groupby="genotype", method='t-test_overestim_var', n_genes=50, copy=True)

<p><strong>Note:</strong> The function <code class="language-plaintext highlighter-rouge">rank_genes_groups</code> does not return a DataFrame that we can use but instead metadata about the marker table, so first we need to construct the marker table using this generated metadata. This is done using the following function, however it’s not too important to understand what this code does!</p>


In [ ]:
def generate_marker_table(adata):
    # extract marker table metadata
    res = adata.uns['rank_genes_groups']

    # generate DataFrame from metadata
    res_df = pd.DataFrame({
                "genes": pd.DataFrame(res["names"]).stack(),
                "scores": pd.DataFrame(res["scores"]).stack(),
                "logfoldchanges": pd.DataFrame(res["logfoldchanges"]).stack(),
                "pvals": pd.DataFrame(res["pvals"]).stack(),
                "pvals_adj": pd.DataFrame(res["pvals_adj"]).stack(),
            })

    # convert row names to columns
    res_df.index.name = 'newhead'
    res_df.reset_index(inplace=True)

    # rename generic column names
    res_df = res_df.rename(columns={'level_0': 'rank', 'level_1':'cluster'})

    # reorder columns
    res_df = res_df.reindex(columns=['cluster', 'rank', 'genes', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj'])

    # insert ref column
    res_df.insert(2, 'ref', 'rest')

    return res_df

<p>Now we can generate our marker tables!</p>


In [ ]:
# Generate marker tables
cluster_marker_table = generate_marker_table(markers_cluster)
genotype_marker_table = generate_marker_table(markers_genotype)

display(cluster_marker_table.head(4))
display(genotype_marker_table.head(4))

<p>Now, there’s a small problem here, which is that if you inspect the output marker tables (above tables), you won’t see gene names, you’ll see Ensembl IDs. While this is a more bioinformatically accurate way of doing this (not every ID has a gene name!), we might want to look at more well-recognised gene names, so let’s pop some of that information in!</p>


In [ ]:
# Join two datasets

cluster_joined = pd.merge(cluster_marker_table, markers_cluster.var, left_on='genes', right_on='ID')
genotype_joined = pd.merge(genotype_marker_table, markers_genotype.var, left_on='genes', right_on='ID')

display(cluster_joined.head(5))
display(genotype_joined.head(5))

In [ ]:
# Cut columns from tables

cluster_markers_named = cluster_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]
genotype_markers_named = genotype_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]

display(cluster_markers_named.head(5))
display(genotype_markers_named.head(5))

<p>Well done! It’s time for the best bit, the plotting!</p>
<h1 id="plotting">Plotting!</h1>
<p>It’s time! Let’s plot it all! But first, let’s pick some marker genes from the <code class="language-plaintext highlighter-rouge">markers_cluster</code> list that you made as well. I’ll be honest, in practice, you’d now be spending a lot of time looking up what each gene does (thank you google!). There are burgeoning automated-annotation tools, however, so long as you have a good reference (a well annotated dataset that you’ll use as the ideal). In the mean time, let’s do this the old-fashioned way, and just copy a bunch of the markers in the original paper.</p>


In [ ]:
# PCA
sc.pl.embedding(
    markers_cluster,
    basis='pca',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# TSNE
sc.pl.embedding(
    markers_cluster,
    basis='tsne',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# UMAP
sc.pl.embedding(
    markers_cluster,
    basis='umap',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

<p>{% icon congratulations %} Congratulations! You now have plots galore!</p>
<h1 id="insights-into-the-beyond">Insights into the beyond</h1>
<p>Now it’s the fun bit! We can see where genes are expressed, and start considering and interpreting the biology of it. At this point, it’s really about what information you want to get from your data - the following is only the tip of the iceberg. However, a brief exploration is good, because it may help give you ideas going forward with for your own data. Let us start interrogating our data!</p>
<h2 id="biological-interpretation">Biological Interpretation</h2>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-appearance-is-everything"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Appearance is everything</div>
<p>Which visualisation is the most useful for getting an overview of our data, <em>pca</em>, <em>tsne</em>, or <em>umap</em>?</p>
<figure id="figure-12" style="max-width: 90%;"><img src="...
"" alt="PCA-tSNE-UMAP. " width="3304" height="1234" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/louvain_clustering.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 12</strong>:</span> Louvain clustering by dimension reduction</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-10"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-10" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>You can see why a PCA is generally not enough to see clusters in samples - keep in mind, you’re only seeing components 1 and 2! - and therefore why the tSNE and UMAP visualisation dimensionality reductions are so useful. But there is not necessarily a clear winner between tSNE and UMAP, but I think UMAP is slightly clearer with its clusters, so we’ll stick with that for the rest of the analysis.</p>
</details>
</blockquote>
<p>Note that the cluster numbering is based on size alone - clusters 0 and 1 are not necessarily related, they are just the clusters containing the most cells. It would be nice to know what exactly these cells are. This analysis (googling all of the marker genes, both checking where the ones you know are as well as going through the marker tables you generated!) is a fun task for any individual experiment, so we’re going to speed past that and nab the assessment from the original paper!</p>
<table>
<thead>
<tr>
<th>Clusters</th>
<th>Marker</th>
<th>Cell type</th>
</tr>
</thead>
<tbody>
<tr>
<td>4</td>
<td>Il2ra</td>
<td>Double negative (early T-cell)</td>
</tr>
<tr>
<td>0,1,2,6</td>
<td>Cd8b1, Cd8a, Cd4</td>
<td>Double positive (middle T-cell)</td>
</tr>
<tr>
<td>5</td>
<td>Cd8b1, Cd8a, Cd4 - high</td>
<td>Double positive (late middle T-cell)</td>
</tr>
<tr>
<td>3</td>
<td>Itm2a</td>
<td>Mature T-cell</td>
</tr>
<tr>
<td>7</td>
<td>Hba-a1</td>
<td>RBCs (as impurity here)</td>
</tr>
<tr>
<td>-</td>
<td>Aif1</td>
<td>Macrophages</td>
</tr>
</tbody>
</table>
<p>The authors weren’t interested in further annotation of the DP cells, so neither are we. Sometimes that just happens. The maths tries to call similar (ish) sized clusters, whether it is biologically relevant or not. Or, the question being asked doesn’t really require such granularity of clusters.</p>
<figure id="figure-13" style="max-width: 90%;"><img src="...
"" alt="Known marker gene locations. " width="1867" height="851" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/known_marker_gene_locations.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 13</strong>:</span> Known marker genes locations</figcaption></figure>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-important"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-important" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Important!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you have deviated from any of the original parameters in this tutorial, you will likely have a different number of clusters. You will, therefore, need to change the <code class="language-plaintext highlighter-rouge">categories</code> parameter in <code class="language-plaintext highlighter-rouge">rename_categories</code> accordingly. Best of luck!</p>
</blockquote>
<h2 id="annotating-clusters">Annotating Clusters</h2>
<p>As mentioned at the beginning of the tutorial, you might not get outputs identical to a tutorial if you are running it in a programming environment, but the outputs should still be pretty close. However, you have to double check if the categories (ie. cell types) in the example below correspond to the identified cluster numbers based on gene expression. You might need to change the order of assigned cell types in the <em>categories</em> parameter to match the cluster numbers identified by <em>louvain</em>.</p>


In [ ]:
# Add meaningful names to each category
markers_cluster.rename_categories(key='louvain',
    categories=['DP-M4','DP-M3','T-mat','DN','DP-M2','DP-M1','DP-L','RBCs'])   # categories (cell types) correspond to the identified cluster numbers, ie. [0, 1, 2,...,7]

# Copy AnnData object
markers_cluster_copy = markers_cluster.copy()

# Rename 'louvain' column
markers_cluster_copy.obs = markers_cluster_copy.obs.rename(columns={'louvain': 'cell_type'})

# Scanpy - plot updated object
sc.pl.embedding(
    markers_cluster_copy,
    basis='umap',
    color=['cell_type','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','Hba-a1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='-annotated.png'
)

<figure id="figure-14" style="max-width: 90%;"><img src="...
"" alt="Annotated Embeddings. " width="589" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/annotated_umap.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 14</strong>:</span> Our Annotated UMAP</figcaption></figure>
<p>Now that we know what we’re dealing with, let’s examine the effect of our variable, proper science!</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-genotype"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Genotype</div>
<p>Are there any differences in genotype? Or in biological terms, is there an impact of growth restriction on T-cell development in the thymus?</p>
<figure id="figure-15" style="max-width: 90%;"><img src="...
HBwcHBxrHP8fVgKajBU24VsAAAAASUVORK5CYII=
"" alt="Genotype Images. " width="1197" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/genotype_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 15</strong>:</span> Genotype differences</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-11"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-11" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We can see that DP-L, which seems to be extending away from the DP-M bunch, as well as the mature T-cells (or particularly the top half) are missing some knockout cells. Perhaps there is some sort of inhibition here? INTERESTING! What next? We might look further at the transcripts present in both those populations, and perhaps also look at the genotype marker table… So much to investigate! But before we set you off to explore to your heart’s delight, let’s also look at this a bit more technically.</p>
</details>
</blockquote>
<h2 id="technical-assessment">Technical Assessment</h2>
<p>Is our analysis real? Is it right? Well, we can assess that a little bit.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch effect</div>
<p>Is there a batch effect?</p>
<figure id="figure-16" style="max-width: 90%;"><img src="...
"" alt="Batch effect. " width="502" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/batch_effect.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 16</strong>:</span> Batch effect?</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-12"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-12" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>While some shifts are expected and nothing to be concerned about, DP-L looks to be mainly comprised of N705. There might be a bit of batch effect, so you could consider using batch correction on this dataset. However, if we focus our attention on the other cluster - mature T-cells -  where there is batch mixing, we can still assess this biologically even without batch correction.
Additionally, we will also look at the confounding effect of sex.</p>
<figure id="figure-17" style="max-width: 90%;"><img src="...
SUVORK5CYII=
"" alt="Sex effect. " width="1623" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/sex_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 17</strong>:</span> Sex differences</figcaption></figure>
<p>We note that the one female sample - unfortunately one of the mere three knockout samples - seems to be distributed in the same areas as the knockout samples at large, so luckily, this doesn’t seem to be a confounding factor and we can still learn from our data. Ideally, this experiment would be re-run with either more female samples all around or swapping out this female from the male sample.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-depth-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Depth effect</div>
<p>Are there any clusters or differences being driven by sequencing depth, a technical and random factor?</p>
<figure id="figure-18" style="max-width: 90%;"><img src="...
BgwY/Lr4/wGs4iqnD4B2+gAAAABJRU5ErkJggg==
"" alt="Sequencing depth. " width="1110" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/counts_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 18</strong>:</span> Counts across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-13"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-13" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Eureka! This explains the odd DP shift between wildtype and knockout cells - the left side of the DP cells simply have a higher sequencing depth (UMIs/cell) than the ones on the right side. Well, that explains some of the sub-cluster that we’re seeing in that splurge. Importantly, we don’t see that the DP-L or (mostly) the mature T-cell clusters are similarly affected. So, whilst again, this variable of sequencing depth might be something to regress out somehow, it doesn’t seem to be impacting our dataset. The less you can regress/modify your data, in general, the better - you want to stay as true as you can to the raw data, and only use maths to correct your data when you really need to (and not to create insights where there are none!).</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-sample-purity"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Sample purity</div>
<p>Do you think we processed these samples well enough?</p>
<figure id="figure-19" style="max-width: 90%;"><img src="...
"" alt="Sequencing depth. " width="407" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/hemoglobin.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 19</strong>:</span> Hemoglobin across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-14"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-14" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We have seen in the previous images that these clusters are not very tight or distinct, so we could consider stronger filtering. Additionally, hemoglobin - a red blood cell marker that should NOT be found in T-cells - appears throughout the entire sample in low numbers. This suggests some background in the media the cells were in, and we might consider in the wet lab trying to get a purer, happier sample, or in the dry lab, techniques such as SoupX or others to remove this background. Playing with filtering settings (increasing minimum counts/cell, etc.) is often the place to start in these scenarios.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-clustering-resolution"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Clustering resolution</div>
<p>Do you think the clustering is appropriate? i.e. are there single clusters that you think should be separate, and multiple clusters that could be combined?</p>
<figure id="figure-20" style="max-width: 90%;"><img src="...
rkJggg==
"" alt="Itm2a Expression. " width="1093" height="423" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/itm2a_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 20</strong>:</span> Itm2a across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-15"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-15" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Important to note, lest all bioinformaticians combine forces to attack the biologists: just because a cluster doesn’t look like a cluster by eye is NOT enough to say it’s not a cluster! But looking at the biology here, we struggled to find marker genes to distinguish the DP population, which we know is also affected by depth of sequencing. That’s a reasonable argument that DP-M1, DP-M2, and DP-M3 might not be all that different. Maybe we need more depth of sequencing across all the DP cells, or to compare these explicitly to each other (consider variations on FindMarkers!). However, DP-L is both seemingly leaving the DP cluster and also has fewer knockout cells, so we might go and look at what DP-L is expressing in the marker genes. If we look at T-mat further, we can see that its marker gene - Itm2a - is only expressed in half of the cluster. You might consider sub-clustering this to investigate further, either through changing the resolution or through analysing this cluster alone.
If we look at the differences between genotypes alone (so the pseudo-bulk), we can see that most of the genes in that list are actually ribosomal. This might be a housekeeping background, this might be cell cycle related, this might be biological, or all three. You might consider investigating the cycling status of the cells, or even regressing this out (which is what the authors did).</p>
</blockquote>
</blockquote>
<p>Ultimately, there are quite a lot ways to analyse the data, both within the confines of this tutorial (the many parameters that could be changed throughout) and outside of it (batch correction, sub-clustering, cell-cycle scoring, inferred trajectories, etc.) Most analyses will still yield the same general output, though: there are fewer knockout cells in the mature T-cell population.</p>
<p>{% icon congratulations %} Congratulations! You have interpreted your plots in several important ways!</p>
<h1 id="export-your-data-and-notebook-and-figures">Export your data, and notebook, and figures</h1>
<p>It’s now time to export your data! First, we need to get Jupyter to see it as a file.</p>


In [ ]:
adata.write('MarkersCluster.h5ad')

<p>Now you can export it.</p>


In [ ]:
put("MarkersCluster.h5ad")

<p>To export your notebook to your Galaxy history, you can use the following. Change the text to be your notebook name. Do not use spaces!</p>


In [ ]:
put("name_of_jupyter_notebook.ipynb")

<p>Want to export some plots? Choose any (or all) of the plots you saved as files in the folder at the left and put their titles in the following. You can run multiple exports at the same time.</p>


In [ ]:
put("figures/plotname.png")
put("figures/plotname.png")
put("figures/plotname.png")

<h1 id="conclusion">Conclusion</h1>
<p>{% icon congratulations %} Congratulations! You’ve made it to the end!</p>
<p>In this tutorial, you moved from technical processing to biological exploration. By analysing real data - both the exciting and the messy! - you have, hopefully, experienced what it’s like to analyse and question a dataset, potentially without clear cut-offs or clear answers. If you were working in a group, you each analysed the data in different ways, and most likely found similar insights. One of the biggest problems in analysing scRNA-seq is the lack of a clearly defined pathway or parameters. You have to make the best call you can as you move through your analysis, and ultimately, when in doubt, try it multiple ways and see what happens!</p>
<p>If, for some reasons anything didn’t work in Galaxy JupyterLab environment, please don’t get discouraged - we prepared a <a href="https://colab.research.google.com/drive/1DkCysA77iaFAWoKJ1vwE5_qYV8Su7UsX?usp=sharing">Google Colab notebook version</a> for you as a backup so that you can enjoy the tutorial no matter what!</p>


# Key Points

- Single cell data is huge, and must have its many (# genes) dimensions reduced for analysis
- Analysis is more subjective than we think, and biological understanding of the samples as well as many iterations of analysis are important to give us our best change of attaining real biological insights

# Congratulations on successfully completing this tutorial!

Please [fill out the feedback on the GTN website](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-case-jupyter_basic-pipeline/tutorial.html#feedback) and check there for further resources!
